# 1 Descargar Ventas y Visitas (aje-prd-analytics-artifacts-s3)

In [1]:
# import psutil
# import time
# import threading
# import matplotlib.pyplot as plt

# # Variables globales
# mem_usage_gb = []
# mem_usage_percent = []
# cpu_usage_gb = []
# cpu_usage_percent = []
# timestamps = []
# running = True  # Variable de control

# # Función de monitoreo en segundo plano
# def monitor_resources():
#     global running
#     start_time = time.time()
    
#     while running:
#         # Obtener información del sistema
#         mem_info = psutil.virtual_memory()
#         cpu_percent = psutil.cpu_percent(interval=0.5)  # Medir CPU sin bloquear
#         cpu_used_gb = (cpu_percent / 100) * (psutil.virtual_memory().total / (1024**3))  # Estimación del uso de CPU en GB
        
#         # Guardar datos
#         timestamp = time.time() - start_time
#         mem_usage_gb.append(mem_info.used / (1024 ** 3))  # RAM usada en GB
#         mem_usage_percent.append(mem_info.percent)  # RAM en %
#         cpu_usage_gb.append(cpu_used_gb)  # CPU en GB
#         cpu_usage_percent.append(cpu_percent)  # CPU en %
#         timestamps.append(timestamp)
        
#         # Imprimir valores en tiempo real
#         # print(f"Tiempo: {timestamp:.2f}s | RAM: {mem_info.percent:.2f}% ({mem_usage_gb[-1]:.2f} GB) | CPU: {cpu_percent:.2f}% ({cpu_used_gb:.2f} GB)")
        
#         time.sleep(1)  # Medir cada 1 segundo

# # Iniciar monitoreo en segundo plano
# monitor_thread = threading.Thread(target=monitor_resources, daemon=True)
# monitor_thread.start()

# print("Monitoreo de CPU y RAM iniciado en segundo plano...")

In [2]:
!pip install --upgrade pip setuptools wheel
!pip install --upgrade pyarrow==17.0.0
!pip install "awswrangler[redshift]" --no-build-isolation
!pip install psycopg2-binary

In [3]:
import pandas as pd
import redshift_connector
import awswrangler as wr
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import os
import boto3
import io
import pytz
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import re
import gc

In [4]:
def comprobar_inputs():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"
    
    hoy = datetime.now(pytz.timezone("America/Lima")).date()
    errores = []

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" not in objetos:
        print("ERROR: No se encontraron archivos en la ruta especificada.")
        return

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        key = objeto["Key"]
        if key == "pedido_sugerido/data-v1/cam/ventas_sellout_cam000":
            print(f"OJO: El key {key} aplica a la validacion.")
            continue
        print(key)
        # Omitir "carpetas" en S3
        if key.endswith("/"):
            continue

        last_modified = objeto["LastModified"].date()
        size_kb = objeto["Size"] / 1024  # Convertir a KB

        # Verificar si el objeto tiene contenido real (FULL_OBJECT)
        if objeto["Size"] == 0:
            errores.append(f"ERROR: El archivo {key} está vacío.")

        # Verificar si el archivo ha sido modificado hoy
        if last_modified != hoy:
            errores.append(f"ERROR: El archivo {key} no ha sido modificado hoy ({hoy}), su ultima fecha de modificacion fue {last_modified}.")

        # Verificar si el tamaño del archivo es menor a 1 KB
        if size_kb < 1:
            errores.append(f"ALERTA: El archivo {key} tiene un tamaño menor a 1 KB ({size_kb:.2f} KB).")

    # Mostrar los errores y lanzar una excepción si es necesario
    if errores:
        for error in errores:
            print(error)
        raise ValueError("Se encontraron problemas con los archivos en S3.")
    else:
        print("Todo bien :D")

In [5]:
comprobar_inputs()

pedido_sugerido/data-v1/cam/
pedido_sugerido/data-v1/cam/D_stock_cr.csv
pedido_sugerido/data-v1/cam/D_stock_gt.csv
pedido_sugerido/data-v1/cam/D_stock_ni.csv
pedido_sugerido/data-v1/cam/D_stock_pa.csv
pedido_sugerido/data-v1/cam/maestro_productos_cam000
pedido_sugerido/data-v1/cam/ventas_cam000
pedido_sugerido/data-v1/cam/ventas_cam001
OJO: El key pedido_sugerido/data-v1/cam/ventas_sellout_cam000 aplica a la validacion.
pedido_sugerido/data-v1/cam/visitas_cam000
Todo bien :D


In [6]:
# Obtener cod SKUs y descripcion de CAM
query=f'''select * from
    comercial_cam.dim_producto
    where estado='A' and cod_compania = '0070';
    '''

con = wr.data_api.redshift.connect(cluster_id = 'dwh-cloud-storage-salesforce-prod', database = 'dwh_prod',db_user='dwhuser')
maestro_prod = wr.data_api.rds.read_sql_query( query, con)

In [7]:
maestro_prod = maestro_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
maestro_prod = maestro_prod.groupby(["cod_articulo_magic"]).first().reset_index()
maestro_prod.to_csv("Input/D_pan_masterProd.csv", index=False)

In [8]:
rutas_test = [
1001,1002,1003,1004,1005,1006,29000,29001,1050,1051,1007,1008,1009,1010,
1011,1012,1601,1602,1603,1605,1150,1152,7010,7012,7015,5010,5011,5012,5013,
6001,6002,6003,6004,6005,1250,1251,1252,1253,1350,1352,1353,1450,1451,
1001,1002,1003,1004,1005,1006,29000,29001,1050,1051,1007,1008,1009,1010,
1011,1012,1601,1602,1603,1606,1150,1152,7010,7012,7015,5010,5011,5012,5013,
6001,6002,6003,6004,6005,1250,1251,1252,1253,1350,1352,1353,1450,1451]


print(rutas_test)

[1001, 1002, 1003, 1004, 1005, 1006, 29000, 29001, 1050, 1051, 1007, 1008, 1009, 1010, 1011, 1012, 1601, 1602, 1603, 1605, 1150, 1152, 7010, 7012, 7015, 5010, 5011, 5012, 5013, 6001, 6002, 6003, 6004, 6005, 1250, 1251, 1252, 1253, 1350, 1352, 1353, 1450, 1451, 1001, 1002, 1003, 1004, 1005, 1006, 29000, 29001, 1050, 1051, 1007, 1008, 1009, 1010, 1011, 1012, 1601, 1602, 1603, 1606, 1150, 1152, 7010, 7012, 7015, 5010, 5011, 5012, 5013, 6001, 6002, 6003, 6004, 6005, 1250, 1251, 1252, 1253, 1350, 1352, 1353, 1450, 1451]


In [9]:
len(rutas_test)

86

In [10]:
clientes_ruta_test = []

In [11]:
def descargar_visitas():
    # Guardar clientes de las rutas
    global clientes_ruta_test

    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0 and objeto["Key"].split("/")[-1] == "visitas_cam000":
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "visitas_cam000":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df.pais__c == "CR")
                    & (df["cod_ruta"].isin(rutas_test))
                    & (df.codigo_canal__c == 2)
                ].reset_index(drop=True)
                
                # Guardamos estos clientes para descargarlos en ventas (por si cambiaron de ruta)
                clientes_ruta_test = df["codigo_cliente__c"].unique()
                nombre_csv = f"Input/{nombre_archivo}.parquet"
                df["personalizacion"] = df["personalizacion"].astype(str)
                df.to_parquet(nombre_csv, index=False)
            else:
                continue

In [12]:
%%time
descargar_visitas()

{'ResponseMetadata': {'RequestId': 'H8AEKKA9AFPV22QS', 'HostId': '3h4BihiAWaaJMcT/NVPNuf2crDLjscHttXYOb2r2xdOcHPZSp6V3eclkDWcaOC+MUOTcAjYvg5EGS+FUfNObK3TH+1xFxBtU', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '3h4BihiAWaaJMcT/NVPNuf2crDLjscHttXYOb2r2xdOcHPZSp6V3eclkDWcaOC+MUOTcAjYvg5EGS+FUfNObK3TH+1xFxBtU', 'x-amz-request-id': 'H8AEKKA9AFPV22QS', 'date': 'Mon, 30 Mar 2026 02:25:42 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/cam/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 33, 15, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/cam/D_stock_cr.csv', 'LastModified': datetime.datetime(2026, 3, 29, 14, 3, 33, tzinfo=tzlocal()), 'ETag': '"7b4bd95b96c01f6280a5e987d040e7d6"', 'ChecksumAlgorithm': ['CRC32'], 'Checksu

In [13]:
def descargar_ventas():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)

    # Creamos DF que juntará los 2 archivos
    final_csv = pd.DataFrame()

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0:
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "ventas_cam000":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df["cod_ruta"].isin(rutas_test))
                        | (df["cod_cliente"].isin(clientes_ruta_test))
                ].reset_index(drop=True)[['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']]
                final_csv = pd.concat([final_csv, df], ignore_index=True)
            elif nombre_archivo == "ventas_cam001":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df["cod_ruta"].isin(rutas_test))
                        | (df["cod_cliente"].isin(clientes_ruta_test))
                ].reset_index(drop=True)[['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']]
                final_csv = pd.concat([final_csv, df], ignore_index=True)

            else:
                continue

    nombre_csv = f"Input/ventas_cam_all.parquet"
    final_csv.to_parquet(nombre_csv, index=False)

In [14]:
%%time
descargar_ventas()

{'ResponseMetadata': {'RequestId': 'HZ5SXX77G4MSFY8J', 'HostId': '5kUSpxQ2Ok/P6PzAY2iKWqeNMayWq+8o9TeZ6+/xAFvjtZ6EDlyighTE2KdYYD/OU5KKa6oLwMvtCipw2Bu2HT1oWVMe10zz', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '5kUSpxQ2Ok/P6PzAY2iKWqeNMayWq+8o9TeZ6+/xAFvjtZ6EDlyighTE2KdYYD/OU5KKa6oLwMvtCipw2Bu2HT1oWVMe10zz', 'x-amz-request-id': 'HZ5SXX77G4MSFY8J', 'date': 'Mon, 30 Mar 2026 02:25:44 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/cam/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 33, 15, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/cam/D_stock_cr.csv', 'LastModified': datetime.datetime(2026, 3, 29, 14, 3, 33, tzinfo=tzlocal()), 'ETag': '"7b4bd95b96c01f6280a5e987d040e7d6"', 'ChecksumAlgorithm': ['CRC32'], 'Checksu

/tmp/ipykernel_7255/2670914792.py:28: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(content), sep=";")


CPU times: user 2min 2s, sys: 21.7 s, total: 2min 24s
Wall time: 4min 17s


## 1.1 Juntar VENTAS y VISITAS

In [15]:
import pandas as pd
from datetime import datetime, timedelta

In [16]:
pan_ventas = pd.read_parquet("Input/ventas_cam_all.parquet")
pan_visitas = pd.read_parquet("Input/visitas_cam000.parquet")

In [17]:
pan_ventas = pan_ventas[pan_ventas.cod_pais=="CR"].reset_index(drop=True)

In [18]:
pan_visitas = pan_visitas[(pan_visitas.codigo_canal__c==2)&(pan_visitas.compania__c=="0070")].reset_index(drop=True)

In [19]:
pan_ventas["cod_articulo_magic"] = pan_ventas["id_producto"].str.split("|").str[-1]
pan_ventas["cod_articulo_magic"] = pan_ventas["cod_articulo_magic"].astype(int)

In [20]:
pan_ventas.shape

(1946385, 26)

In [21]:
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print(hoy)
# Filtrar solo las filas con la fecha de hoy
pan_ventas = pan_ventas[pan_ventas['fecha_proceso'] == hoy]

20260329


In [22]:
pan_ventas.shape

(1946385, 26)

In [23]:
# Obtener la fecha y año actual
current_date = datetime.now(pytz.timezone("America/Lima"))
# Formatear la fecha en el formato "YYYYMM"
formatted_date = current_date.strftime("%Y-%m-%d")
formatted_date

'2026-03-29'

In [24]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_CostaRica/Input/visitas_cam000_{formatted_date}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(pan_visitas, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_CostaRica/Input/visitas_cam000_2026-03-29.csv'],
 'partitions_values': {}}

In [25]:
PS_VAL_total_skus_venta = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_venta

85

In [26]:
# PAN VENTAS
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_ventas["id_cliente"] = (
    "CAM"
    + "|"
    + pan_ventas["cod_compania"].astype(str)
    + "|"
    + pan_ventas["cod_cliente"].astype(str)
)

# PAN VISITAS
pan_visitas["compania__c"] = (
    pan_visitas["compania__c"].astype(str).apply(lambda x: str((x)).rjust(4, "0"))
)
pan_visitas["id_cliente"] = (
    "CAM"
    + "|"
    + pan_visitas["compania__c"].astype(str)
    + "|"
    + pan_visitas["codigo_cliente__c"].astype(str)
)

In [27]:
pan_visitas["dias_de_visita__c"].value_counts(dropna=False)

dias_de_visita__c
1      2205
4      2108
2      2087
5      1974
3      1934
6      1760
3;6    1484
2;5    1354
1;4    1264
3;7      69
7        53
1;6      13
2;6      12
1;5      10
3;4       4
4;5       3
2;4       3
1;3       2
4;6       2
1;7       1
Name: count, dtype: int64

In [28]:
pan_ventas.shape

(1946385, 26)

In [29]:
pan_ventas["fecha_liquidacion"].sort_values().unique()[-10:]

array(['2026-03-17', '2026-03-18', '2026-03-19', '2026-03-20',
       '2026-03-21', '2026-03-23', '2026-03-24', '2026-03-25',
       '2026-03-26', '2026-03-27'], dtype=object)

In [30]:
PS_VAL_ultima_venta = pan_ventas["fecha_liquidacion"].sort_values().unique()[-1:]
PS_VAL_ultima_venta

array(['2026-03-27'], dtype=object)

In [31]:
pan_visitas["ultima_visita"].sort_values().unique()[-10:]

array(['2026-03-19', '2026-03-20', '2026-03-21', '2026-03-23',
       '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27',
       '2026-03-28', None], dtype=object)

In [32]:
PS_VAL_ultima_visita = pan_visitas["ultima_visita"].sort_values().unique()[-2:-1]
PS_VAL_ultima_visita

array(['2026-03-28'], dtype=object)

In [33]:
PS_VAL_total_clientes_venta = pan_ventas["id_cliente"].nunique()
PS_VAL_total_clientes_venta

11226

In [34]:
PS_VAL_total_clientes_visita = pan_visitas["id_cliente"].nunique()
PS_VAL_total_clientes_visita

11129

In [35]:
pan_visitas = pan_visitas[pan_visitas.codigo_canal__c == 2].reset_index(drop=True)

In [36]:
PS_VAL_total_clientes_visita_trad = pan_visitas["id_cliente"].nunique()
PS_VAL_total_clientes_visita_trad

11129

In [37]:
# Eliminar Duplicados y quedarse con la ultima visita más reciente
visita_default_semana_atras = (datetime.now(pytz.timezone("America/Lima")) - timedelta(days=7)).strftime("%Y-%m-%d")
pan_visitas["ultima_visita"] = pan_visitas["ultima_visita"].fillna(
    visita_default_semana_atras
)
pan_visitas = (
    pan_visitas.sort_values(["id_cliente", "ultima_visita"], ascending=False)
    .groupby("id_cliente")
    .head(1)
)

In [38]:
# Quitamos cod_modulo de ventas para luego quedarnos con el de visitas
# pan_ventas = pan_ventas.drop(["cod_modulo","cod_ruta"], axis=1)

In [39]:
# pan_ventas=pd.merge(pan_ventas,pan_visitas[["id_cliente","dias_de_visita__c","periodo_de_visita__c","ultima_visita","cod_ruta","cod_modulo"]],on=["id_cliente"],how="outer").drop_duplicates()

In [40]:
# Combina ventas y visitas (nos quedamos con ruta y módulo de visitas si es nulo, usamos el valor de ventas)
df_merged = pd.merge(
    pan_ventas,
    pan_visitas[
        [
            "id_cliente",
            "dias_de_visita__c",
            "periodo_de_visita__c",
            "ultima_visita",
            "cod_ruta",
            "cod_modulo",
            "eje_potencial__c",
        ]
    ],
    on="id_cliente",
    # how='outer',  # Este es un merge 'outer', para conservar todos los clientes de ambos DataFrames.
    how="inner",
    suffixes=(
        "_df1",
        "_df2",
    ),  # Añadimos sufijos para diferenciar las columnas que se solapan
)

# Reemplazar los valores nulos de 'cod_ruta_df2' con los valores de 'cod_ruta_df1', y viceversa
# Si 'cod_ruta_df2' es nulo, se usará el valor de 'cod_ruta_df1'. Si ambos son nulos, se mantendrá nulo.
df_merged["cod_ruta"] = df_merged["cod_ruta_df2"].combine_first(
    df_merged["cod_ruta_df1"]
)
df_merged["cod_modulo"] = df_merged["cod_modulo_df2"].combine_first(
    df_merged["cod_modulo_df1"]
)

# Eliminar las columnas innecesarias ('cod_ruta_df1', 'cod_ruta_df2', 'cod_modulo_df1', 'cod_modulo_df2')
df_merged = df_merged.drop(
    columns=["cod_ruta_df1", "cod_ruta_df2", "cod_modulo_df1", "cod_modulo_df2"]
)

In [41]:
pan_ventas = df_merged.copy()
# Liberar memoria del df_merged
df_merged = None 
gc.collect()

1344

In [42]:
pan_ventas["cod_cliente"] = pan_ventas["cod_cliente"].astype(str)
pan_ventas["cod_ruta"] = pan_ventas["cod_ruta"].astype(int)
pan_ventas["cod_modulo"] = pan_ventas["cod_modulo"].astype(int)

In [43]:
# Al cruzar visita con ventas. hay clientes de. visita que no tienen informacion de venta, esto se remplaza a nivel de ruta.
# si la ruta entera no tiene ventas, se ignora
# Este df guarda clientes con visitas y sin ventas para luego recomendarlos por separado
df_clientes_con_visitas_sin_ventas = pd.DataFrame()
for ruta in pan_ventas["cod_ruta"].unique():
    temp_df = pan_ventas[pan_ventas["cod_ruta"] == ruta]
    # Si la ruta entera no tiene ninguna venta, se guarda en un dataframe aparte
    if len(temp_df) == (temp_df["cod_articulo_magic"].isnull().sum()):
        df_clientes_con_visitas_sin_ventas = pd.concat(
            [df_clientes_con_visitas_sin_ventas, temp_df], axis=0
        ).reset_index(drop=True)

In [44]:
pan_ventas["id_cliente"].nunique()

10886

In [45]:
PS_VAL_total_clientes_merged = pan_ventas["id_cliente"].nunique()
PS_VAL_total_clientes_merged

10886

In [46]:
PS_VAL_total_skus_merged = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_merged

85

In [47]:
PS_VAL_total_fechas_unicas = pan_ventas.fecha_liquidacion.sort_values().nunique()
PS_VAL_total_fechas_unicas

338

In [48]:
pan_ventas.groupby("cod_sucursal").agg(
    {"cod_articulo_magic": "nunique", "id_cliente": "nunique", "cod_ruta": "nunique"}
)

,cod_articulo_magic,id_cliente,cod_ruta
cod_sucursal,,,
2,67,5643,25
4,71,992,4
6,71,1228,5
7,73,1250,4
9,74,1607,5
10,55,1922,10


# 2 Ajustar DF de clientes: Se filtrará a los clientes que tienen visita al día siguiente

In [49]:
# CAMBIAR DEPORADE POR SPORADE
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].str.strip()
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].replace({"DEPORADE": "SPORADE"})

pan_prod = pd.read_csv("Input/D_pan_masterProd.csv")

# PAN SEGMENTOS
pan_ventas = pan_ventas.rename(columns={"eje_potencial__c": "new_segment"})
# mapear valores
mapping = {
    "S1": "BLINDAR",
    "S2": "DESARROLLAR",
    "S4": "MANTENER",
    "S5": "OPTIMIZAR",
}

pan_ventas["new_segment"] = pan_ventas["new_segment"].map(mapping)
pan_ventas["new_segment"] = pan_ventas["new_segment"].fillna("OPTIMIZAR")

In [50]:
pan_ventas = pd.merge(
    pan_ventas,
    pan_prod[["cod_articulo_magic", "desc_articulo"]],
    how="left",
    on="cod_articulo_magic",
)

pan_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"}, inplace=True)
pan_ventas["mes"] = pd.to_datetime(pan_ventas["fecha_liquidacion"]).dt.strftime(
    "%Y-%m-01"
)

In [51]:
pan_ventas.to_parquet("Processed/ventas_costarica_12m.parquet", index=False)

## 2.3 Seleccionando SKUs que no se commpran en los ultimos 3 dias
La variable **sucursal_sku** contiene todos los SKUs disponibles por sucursal

In [52]:
# pan_ventas[["cod_compania","cod_sucursal"]].drop_duplicates().to_csv("CR_compa_sucursal_2025-08-01.csv",index=False)

In [53]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [54]:
sucursales = pan_ventas["cod_sucursal"].unique()
sucursales

array([10,  2,  6,  9,  4,  7])

In [55]:
# Obtener la fecha actual
fecha_actual = datetime.now(pytz.timezone("America/Lima")).date()
# Lista para almacenar las fechas
last_3_days = []
dias_atras = 3
sucursal_sku = {}
# Obtener las fechas de los últimos 3 días
for i in range(1, dias_atras + 1):
    fecha = fecha_actual - timedelta(days=i)
    fecha_formato = fecha.strftime("%Y-%m-%d")
    last_3_days.append(fecha_formato)

In [56]:
last_3_days

['2026-03-28', '2026-03-27', '2026-03-26']

In [57]:
for sucursal in sucursales:
    temp = pan_ventas[pan_ventas["cod_sucursal"] == sucursal]
    # sku_fecha=temp[["cod_articulo_magic","fecha_liquidacion"]].sort_values(["cod_articulo_magic","fecha_liquidacion"]).drop_duplicates().reset_index(drop=True)
    # sku_fecha=sku_fecha.groupby("cod_articulo_magic")[["cod_articulo_magic","fecha_liquidacion"]].tail(3)
    # #Listando SKUs comprados al menos una vez en los ultimos 3 dias
    # sku_in_last_3_days=sku_fecha[sku_fecha.fecha_liquidacion.isin(last_3_days)]["cod_articulo_magic"].unique()
    # print(f"Sucursal {sucursal} eliminando SKUs:",sku_fecha[(~sku_fecha["cod_articulo_magic"].isin(sku_in_last_3_days))].reset_index(drop=True)["cod_articulo_magic"].sort_values().unique())
    # #SKU COMPRADO AL MENOS una vez en los ultimos 3 dias
    # sku_fecha=sku_fecha[sku_fecha["cod_articulo_magic"].isin(sku_in_last_3_days)].reset_index(drop=True)
    # temp=temp[temp["cod_articulo_magic"].isin(sku_fecha["cod_articulo_magic"].unique())].reset_index(drop=True)
    sucursal_sku[sucursal] = temp["cod_articulo_magic"].sort_values().unique()

## 2.4 Filtrar clientes a visitar mañana

In [58]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [59]:
# Filtrar ventas mayores a 0
# pan_ventas=pan_ventas[pan_ventas["imp_netovta"]>0]

# En lugar de omitir estos clientes, mejor los consideramos como OPTIMIZAR
# pan_ventas.loc[pan_ventas["imp_netovta"] <= 0, "new_segment"] = "OPTIMIZAR"

In [60]:
PS_VAL_total_clientes_imp_netovta_positivo = pan_ventas.id_cliente.nunique()
PS_VAL_total_clientes_imp_netovta_positivo

10886

In [61]:
pan_ventas.dias_de_visita__c.unique()

array(['1;4', '3', '5', '2;5', '6', '4', '1', '3;6', '2'], dtype=object)

In [62]:
pan_ventas.dias_de_visita__c.value_counts(dropna=False)

dias_de_visita__c
3;6    311165
2;5    271335
1;4    262560
4      186898
2      185254
1      184483
5      181659
3      174127
6      169599
Name: count, dtype: int64

In [63]:
data_test = (
    pan_ventas[
        ["id_cliente", "dias_de_visita__c", "periodo_de_visita__c", "ultima_visita"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
data_test["ultima_visita"] = pd.to_datetime(
    data_test["ultima_visita"], format="%Y-%m-%d"
)
# Calcula la diferencia en días entre la fecha de cada fila y la fecha de hoy
fecha_actual = datetime.now() + timedelta(days=0)
data_test["dias_pasados"] = (fecha_actual - data_test["ultima_visita"]).dt.days
fecha_actual

datetime.datetime(2026, 3, 30, 2, 30, 35, 337435)

In [64]:
# Obtener el día de la semana actual (1 para lunes, 2 para martes, ..., 7 para domingo)
dia_actual = datetime.now(pytz.timezone("America/Lima")).weekday() + 1

# Si hoy es domingo (7), el día siguiente es lunes (1), de lo contrario, es el siguiente día al actual
if dia_actual == 6:
    dia_siguiente = 7
else:
    dia_siguiente = (dia_actual + 1) % 7
    # dia_siguiente = (dia_actual ) % 7

# Filtrar los clientes que serán visitados mañana
clientes_a_visitar_manana = data_test[
    data_test["dias_de_visita__c"]
    .astype(str)
    .apply(lambda x: str(dia_siguiente) in x.split(";"))
].reset_index(drop=True)
print("dia_actual", dia_actual)
print("dia_siguiente", dia_siguiente)

dia_actual 7
dia_siguiente 1


In [65]:
# Definir condiciones de filtro
condicion_f1 = clientes_a_visitar_manana["periodo_de_visita__c"] == "F1"
condicion_f2 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F2") & (
    clientes_a_visitar_manana["dias_pasados"] > 13
)
condicion_f3 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F3") & (
    clientes_a_visitar_manana["dias_pasados"] > 20
)
condicion_f4 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F4") & (
    clientes_a_visitar_manana["dias_pasados"] > 27
)

# Aplicar las condiciones de filtro
clientes_a_visitar_manana = clientes_a_visitar_manana[
    condicion_f1 | condicion_f2 | condicion_f3 | condicion_f4
].reset_index(drop=True)

In [66]:
clientes_a_visitar_manana.isnull().sum()

id_cliente              0
dias_de_visita__c       0
periodo_de_visita__c    0
ultima_visita           0
dias_pasados            0
dtype: int64

In [67]:
clientes_a_visitar_manana.periodo_de_visita__c.unique()

array(['F1', 'F2'], dtype=object)

In [68]:
clientes_a_visitar_manana.dias_de_visita__c.unique()

array(['1;4', '1'], dtype=object)

In [69]:
pan_ventas = pan_ventas[
    pan_ventas["id_cliente"].isin(clientes_a_visitar_manana["id_cliente"])
].reset_index(drop=True)

In [70]:
pan_ventas.periodo_de_visita__c.unique()

array(['F1', 'F2'], dtype=object)

In [71]:
pan_ventas.dias_de_visita__c.unique()

array(['1;4', '1'], dtype=object)

In [72]:
PS_VAL_total_clientes_para_manhana = pan_ventas.id_cliente.nunique()
PS_VAL_total_clientes_para_manhana

2161

In [73]:
PS_VAL_total_skus_venta_para_manhana = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_venta_para_manhana

84

In [74]:
pan_ventas.groupby("cod_sucursal").cod_articulo_magic.nunique()

cod_sucursal
2     65
4     71
6     70
7     69
9     74
10    52
Name: cod_articulo_magic, dtype: int64

In [75]:
pan_ventas.to_parquet("Processed/cr_ventas_manana.parquet", index=False)

# 3. Pre Procesamiento

In [76]:
import pandas as pd
import numpy as np
from datetime import datetime

In [77]:
df_ventas = pd.read_parquet("Processed/cr_ventas_manana.parquet")

In [78]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(
    df_ventas["fecha_liquidacion"], format="%Y-%m-%d"
)
df_ventas["desc_marca"] = df_ventas["desc_marca"].str.strip()
df_ventas["desc_categoria"] = df_ventas["desc_categoria"].str.strip()

## 3.1 Mapeo de Pesos por Giro
Los pesos representan el orden en el que se recomendara la marca para cada cliente dependiendo del Giro

In [79]:
# Mapeo de pesos para ordenar las recomendaciones finales segun su importancia por giro
mapeo_pesos = {}
mapeo_diccionario = {}

In [80]:
df_ventas.desc_marca.unique()

array(['BIG', 'VOLT GO', 'CIFRUT', 'PULP', 'BIO ALOE', 'DEST', 'VOLT',
       'CIELO', 'DILYTE', 'D GUSSTO', 'SPORADE', 'KR', 'BIO AMAYU'],
      dtype=object)

In [81]:
df_ventas.desc_categoria.unique()

array(['GASEOSAS', 'ENERGIZANTE', 'JUGOS LIGEROS', 'NECTAR', 'DETERGENTE',
       'AGUA', 'ALOE VERA', 'SUERO', 'CONSERVA DE PESCADO',
       'BEBIDAS DEPORTIVAS', 'CEREALES', 'SOPAS', 'SNACKS', 'PASTAS'],
      dtype=object)

In [82]:
for giro_v in df_ventas["desc_subgiro"].unique():
    temp = df_ventas[(df_ventas["desc_subgiro"] == giro_v)]
    ranks = temp.groupby("desc_categoria")["cant_cajafisicavta"].sum().reset_index()
    ranks.columns = ["index","desc_categoria"]
    ranks = ranks.sort_values(by="desc_categoria", ascending=False)
    if len(list(ranks["desc_categoria"])) <= 5:
        ranks["Ranking"] = range(1, len(ranks) + 1)
    elif len(list(ranks["desc_categoria"])) > 5:
        a = list(ranks["desc_categoria"])
        b = [1, 1, 2, 2]
        # Calculamos el multiplicador para el mapeo de pesos segun la varianza
        if np.std(a) / np.mean(a) <= 1.2:
            multiplicador = 4
        else:
            multiplicador = 2
        if len(a) > 5:
            for i in range(4, len(a)):
                if a[3] <= a[i] * multiplicador:
                    b.append(3)
                else:
                    b.append(3 + i)
        ranks["Ranking"] = b

    print("*" * 20)
    print("GIRO: ", giro_v)
    print("Categorias: ", list(ranks["index"]))
    print("Counts: ", list(ranks["desc_categoria"]))
    mapeo_diccionario[giro_v] = ranks.set_index("index")["Ranking"].to_dict()
    print(ranks.set_index("index")["Ranking"].to_dict())

********************
GIRO:  BODEGAS, TIENDAS
Categorias:  ['GASEOSAS', 'JUGOS LIGEROS', 'ENERGIZANTE', 'AGUA', 'BEBIDAS DEPORTIVAS', 'ALOE VERA', 'SUERO', 'SOPAS', 'SNACKS', 'NECTAR', 'CONSERVA DE PESCADO', 'DETERGENTE', 'CEREALES', 'PASTAS']
Counts:  [233718.9833333333, 43119.38333333333, 18482.166666666664, 6351.733333333313, 5772.416666666666, 4050.0, 2204.1666666666665, 2119.3333333333294, 1457.6666666666633, 1328.0, 1138.0, 872.2499999999754, 434.333333333326, 278.5833333333331]
{'GASEOSAS': 1, 'JUGOS LIGEROS': 1, 'ENERGIZANTE': 2, 'AGUA': 2, 'BEBIDAS DEPORTIVAS': 3, 'ALOE VERA': 3, 'SUERO': 9, 'SOPAS': 10, 'SNACKS': 11, 'NECTAR': 12, 'CONSERVA DE PESCADO': 13, 'DETERGENTE': 14, 'CEREALES': 15, 'PASTAS': 16}
********************
GIRO:  MINIMARKET INDEPENDIENTE
Categorias:  ['GASEOSAS', 'JUGOS LIGEROS', 'ENERGIZANTE', 'AGUA', 'BEBIDAS DEPORTIVAS', 'ALOE VERA', 'SUERO', 'SOPAS', 'SNACKS', 'NECTAR', 'CONSERVA DE PESCADO', 'DETERGENTE', 'CEREALES', 'PASTAS']
Counts:  [230977.308333333

## 3.2. Filtro de SKU por Ruta
Cada Ruta tiene una cantidad de SKUs distintos, la recomendacion será distinta para cada ruta, sin embargo, aquellas rutas que tengan pocos SKUs serán tratadas como una sola ruta

In [83]:
rutas = (
    df_ventas.groupby(["cod_ruta"])["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()["cod_ruta"]
    .unique()
)
rutas

array([ 1010,  1009,  1008,  6001,  1005,  1011,  1001,  1007,  1004,
        1450,  5011,  5012,  1003,  1250,  1002,  1012,  6005,  6002,
        1353,  1051, 29001,  6004,  1251,  1352,  6003, 29000,  1451,
        1152,  1253,  1006,  7012,  1350,  1050,  7015,  1602,  1252,
        1603,  5013,  1601,  1606,  1150,  7010,  5010])

In [84]:
low_sku_ruta = []

**Para rutas que tengan más de 45 SKUS**

In [85]:
for ruta in rutas:
    print("*" * 21)
    print("Ruta:", ruta)
    temp = df_ventas[(df_ventas["cod_ruta"] == ruta)]
    print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
    # print(f"Giros en ruta {ruta}:")
    # print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
    if temp["cod_articulo_magic"].nunique() <= 5:
        low_sku_ruta.append(ruta)
    else:
        temp.to_csv(f"Processed/rutas/D_{ruta}_ventas.csv", index=False)

*********************
Ruta: 1010
SKUs disponibles: 63
*********************
Ruta: 1009
SKUs disponibles: 61
*********************
Ruta: 1008
SKUs disponibles: 62
*********************
Ruta: 6001
SKUs disponibles: 68
*********************
Ruta: 1005
SKUs disponibles: 61
*********************
Ruta: 1011
SKUs disponibles: 62
*********************
Ruta: 1001
SKUs disponibles: 63
*********************
Ruta: 1007
SKUs disponibles: 59
*********************
Ruta: 1004
SKUs disponibles: 62
*********************
Ruta: 1450
SKUs disponibles: 57
*********************
Ruta: 5011
SKUs disponibles: 68
*********************
Ruta: 5012
SKUs disponibles: 66
*********************
Ruta: 1003
SKUs disponibles: 61
*********************
Ruta: 1250
SKUs disponibles: 55
*********************
Ruta: 1002
SKUs disponibles: 63
*********************
Ruta: 1012
SKUs disponibles: 62
*********************
Ruta: 6005
SKUs disponibles: 71
*********************
Ruta: 6002
SKUs disponibles: 68
*********************
Ruta: 

**Para rutas que tengan menos de 45 SKUS**

In [86]:
print("*" * 21)
print("Rutas:", low_sku_ruta)
temp = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
# print(f"Giros en ruta {ruta}:")
# print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
temp.to_csv("Processed/rutas/D_low_ruta_ventas.csv", index=False)

*********************
Rutas: []
SKUs disponibles: 0


# 4. Modelo Pedido Sugerido

## 4.1. Importando librerias necesarias

In [87]:
import os

from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType
from pyspark.sql.functions import (
    when,
    col,
    regexp_replace,
    concat,
    countDistinct,
    lit,
    monotonically_increasing_id,
    hash,
)

from pyspark.ml.linalg import Vector
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [88]:
!java -version

openjdk version "11.0.30" 2026-01-20 LTS
OpenJDK Runtime Environment Corretto-11.0.30.7.1 (build 11.0.30+7-LTS)
OpenJDK 64-Bit Server VM Corretto-11.0.30.7.1 (build 11.0.30+7-LTS, mixed mode)


In [89]:
import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

## 4.2 Construccion del algoritmo ALS

In [90]:
# Se generan csv en Output/Panama/4-resultados/
def als_training_job(cedi, sku_len):

    spark = (
        SparkSession.builder.config("spark.memory.offHeap.enabled", "true")
        .config("spark.memory.offHeap.size", "10G")
        .appName("K-Means for ALS")
        .getOrCreate()
    )

    ventas = (
        spark.read.format("csv")
        .options(header=True, delimiter=",")
        .load(f"Processed/rutas/D_{cedi}_ventas.csv")
    )

    ventas = ventas.na.drop(subset=["fecha_liquidacion"])
    ventas = ventas.groupBy(
        ["id_cliente", "cod_articulo_magic", "cod_compania", "cod_cliente"]
    ).agg(countDistinct("fecha_liquidacion"))
    ventas = ventas.select(
        col("id_cliente").alias("id_cliente"),
        col("count(fecha_liquidacion)").alias("frecuencia"),
        col("cod_articulo_magic").alias("cod_articulo_magic"),
        col("cod_compania").alias("cod_compania"),
        col("cod_cliente").alias("cod_cliente"),
    )

    input_cols = ["frecuencia"]
    vec_assembler = VectorAssembler(inputCols=input_cols, outputCol="features")
    ventas_kmeans = vec_assembler.transform(ventas)
    ventas_kmeans = ventas_kmeans.fillna(0.0, subset=["frecuencia"])

    kmeans = KMeans(featuresCol="features", k=5)
    model = kmeans.fit(ventas_kmeans)

    resumen_ventas = model.transform(ventas_kmeans).groupby("prediction").max()
    resumen_ventas.orderBy(["max(frecuencia)"], ascending=0)
    resumen_ventas = resumen_ventas.select(
        col("prediction").alias("prediction"), col("max(frecuencia)").alias("max")
    )

    maxes = resumen_ventas.select("max").toPandas()["max"]
    maxes = maxes.astype(float)
    maxes = list(maxes)

    labels = [5, 4, 3, 2, 1]
    rank_map = dict(zip(labels, maxes))

    ventas = ventas.withColumn("frecuencia", col("frecuencia").cast(DoubleType()))
    ventas = ventas.withColumn(
        "rating",
        when(ventas["frecuencia"] < rank_map.get(labels[-1]), labels[-1])
        .when(ventas["frecuencia"] < rank_map.get(labels[-2]), labels[-2])
        .when(ventas["frecuencia"] < rank_map.get(labels[-3]), labels[-3])
        .when(ventas["frecuencia"] < rank_map.get(labels[-4]), labels[-4])
        .otherwise(labels[-5]),
    )

    cols = ["id_cliente", "cod_compania", "cod_cliente", "cod_articulo_magic", "rating"]
    als_records = ventas.select(*cols)

    als_records = als_records.withColumn(
        "clienteId", concat("cod_compania", "cod_cliente")
    )
    als_records = als_records.withColumn(
        "clienteId", col("clienteId").cast(IntegerType())
    )
    als_records = als_records.withColumn(
        "cod_articulo_magic", col("cod_articulo_magic").cast(IntegerType())
    )

    als_records = als_records.na.drop(subset=["clienteId", "cod_articulo_magic"])

    input_cols = ["ClienteId", "cod_articulo_magic", "rating"]
    als_records = als_records.select(*input_cols)

    als = ALS(
        rank=15,
        maxIter=5,
        implicitPrefs=True,
        ratingCol="rating",
        itemCol="cod_articulo_magic",
        userCol="ClienteId",
    )
    model = als.fit(als_records)

    # obtener matrices de descomposicion (usuario y articulos)
    user_matrix = model.userFactors
    item_matrix = model.itemFactors

    user_matrix_pd = user_matrix.toPandas()
    item_matrix_pd = item_matrix.toPandas()

    recs = model.recommendForAllUsers(sku_len)
    recs = recs.select("ClienteId", "recommendations.cod_articulo_magic")
    recs_to_parse = recs.toPandas()
    # print(recs_to_parse)

    lista_rec = [f"r{i+1}" for i in range(sku_len)]

    recs_to_parse[lista_rec] = pd.DataFrame(
        recs_to_parse["cod_articulo_magic"].tolist(), index=recs_to_parse.index
    )
    client_recs = pd.melt(
        recs_to_parse, id_vars=["ClienteId"], value_vars=lista_rec
    )

    client_recs["compania"] = client_recs["ClienteId"].apply(lambda x: str(x)[:2])
    client_recs["cliente"] = client_recs["ClienteId"].apply(lambda x: str(x)[2:])
    client_recs["compania"] = client_recs["compania"].apply(lambda x: x.rjust(4, "0"))
    client_recs["id_cliente"] = client_recs["compania"] + "|" + client_recs["cliente"]
    client_recs["cod_articulo_magic"] = client_recs["value"]

    client_recs = (
        client_recs[["id_cliente", "cod_articulo_magic"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Obtener el nombre del archivo CSV
    nombre_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo ya existe
    try:
        # Intentar abrir el archivo para lectura
        with open(nombre_archivo, "r") as f:
            # Si el archivo existe, header=False para no escribir cabeceras
            client_recs.to_csv(
                nombre_archivo, mode="a", header=False, index=False, sep=";"
            )
    except FileNotFoundError:
        # Si el archivo no existe, header=True para escribir las cabeceras
        client_recs.to_csv(nombre_archivo, mode="a", header=True, index=False, sep=";")

    # client_recs.to_csv("Output/Panama/rutas/D_rutas_rec.csv", mode='a', header=False, index=False,sep=";")

## 4.3 Corriendo PS

In [91]:
def runALS():
    # Ruta del archivo que deseas verificar y eliminar si existe
    ruta_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo existe
    if os.path.exists(ruta_archivo):
        # Eliminar el archivo si existe
        os.remove(ruta_archivo)
        print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
    else:
        print(f"El archivo en la ruta {ruta_archivo} no existe.")
    # CORRIENDO ALS PARA RUTAS DE MAS DE 45 SKUs
    for ruta in rutas:
        print("*" * 21)
        temp = pd.read_csv(f"Processed/rutas/D_{ruta}_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en ruta {ruta}:", sku_len)
        als_training_job(ruta, sku_len)
    # CORRIENDO ALS PARA rutas que tengan menos de 45 SKUS
    if len(low_sku_ruta) != 0:
        temp = pd.read_csv(f"Processed/rutas/D_low_ruta_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en low ruta {low_sku_ruta}:", sku_len)
        als_training_job("low_ruta", sku_len)
    else:
        print("No existen rutas con pocos skus :D")

In [92]:
rutas = rutas[~pd.Series(rutas).isin(pd.Series(low_sku_ruta))]
rutas

array([ 1010,  1009,  1008,  6001,  1005,  1011,  1001,  1007,  1004,
        1450,  5011,  5012,  1003,  1250,  1002,  1012,  6005,  6002,
        1353,  1051, 29001,  6004,  1251,  1352,  6003, 29000,  1451,
        1152,  1253,  1006,  7012,  1350,  1050,  7015,  1602,  1252,
        1603,  5013,  1601,  1606,  1150,  7010,  5010])

In [93]:
runALS()

El archivo en la ruta Output/D_rutas_rec.csv ha sido eliminado.
*********************
SKUs disponibles en ruta 1010: 63


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/30 02:30:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/30 02:31:00 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/30 02:31:00 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
26/03/30 02:31:00 WARN InstanceBuilder$NativeLAPACK: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


*********************
SKUs disponibles en ruta 1009: 61
*********************
SKUs disponibles en ruta 1008: 62
*********************
SKUs disponibles en ruta 6001: 68
*********************
SKUs disponibles en ruta 1005: 61
*********************
SKUs disponibles en ruta 1011: 62
*********************
SKUs disponibles en ruta 1001: 63
*********************
SKUs disponibles en ruta 1007: 59
*********************
SKUs disponibles en ruta 1004: 62
*********************
SKUs disponibles en ruta 1450: 57
*********************
SKUs disponibles en ruta 5011: 68
*********************
SKUs disponibles en ruta 5012: 66
*********************
SKUs disponibles en ruta 1003: 61
*********************
SKUs disponibles en ruta 1250: 55
*********************
SKUs disponibles en ruta 1002: 63
*********************
SKUs disponibles en ruta 1012: 62
*********************
SKUs disponibles en ruta 6005: 71
*********************
SKUs disponibles en ruta 6002: 68
*********************
SKUs disponibles en ruta 1

# 5. Ajustes al output del algoritmo ALS

## 5.-9 Quedarnos con skus que hayan tenido ventas en los ultimos 30 dias

In [94]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [95]:
PS_VAL_promedio_recs_por_cliente_modelo = (
    pan_rec.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_modelo

62.344285053216105

In [96]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [97]:
pan_rec

,id_cliente,cod_articulo_magic,cod_ruta
0,CAM|0070|82880,522064,1010
1,CAM|0070|58370,517507,1010
2,CAM|0070|3900,517507,1010
3,CAM|0070|68770,623192,1010
4,CAM|0070|49570,523133,1010
...,...,...,...
134721,CAM|0070|57467,621998,5010
134722,CAM|0070|95807,522502,5010
134723,CAM|0070|65548,623193,5010
134724,CAM|0070|66449,621998,5010


In [98]:
df_ventas = pd.read_parquet("Processed/cr_ventas_manana.parquet")

In [99]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(df_ventas["fecha_liquidacion"])

In [100]:
# Definir el umbral de tiempo (últimos 30 días desde hoy)
# bajamos de 30 a 14 dias 2025-07-25
hoy = pd.Timestamp.today()
fecha_limite = hoy - pd.Timedelta(days=14)
print(fecha_limite)
# Filtrar ventas en los últimos 30 días
ventas_filtradas = df_ventas[df_ventas["fecha_liquidacion"] >= fecha_limite]

2026-03-16 02:32:56.235239


In [101]:
# Obtener los productos vendidos en cada ruta
productos_por_ruta = ventas_filtradas.groupby("cod_ruta")["cod_articulo_magic"].unique().reset_index()

In [102]:
# Unir con los productos vendidos en cada ruta
recomendaciones_validas = pan_rec.merge(productos_por_ruta, on="cod_ruta", how="inner")

In [103]:
recomendaciones_validas.columns = ['id_cliente', 'cod_articulo_magic', 'cod_ruta','lista_sku_ruta']

In [104]:
recomendaciones_validas.shape

(134726, 4)

In [105]:
recomendaciones_validas  = recomendaciones_validas[recomendaciones_validas.apply(lambda row: row["cod_articulo_magic"] in row["lista_sku_ruta"], axis=1)].reset_index(drop=True)

In [106]:
recomendaciones_validas.shape

(99484, 4)

In [107]:
recomendaciones_validas.head()

,id_cliente,cod_articulo_magic,cod_ruta,lista_sku_ruta
0,CAM|0070|82880,522064,1010,"[523134, 522066, 517192, 524092, 523132, 52206..."
1,CAM|0070|58370,517507,1010,"[523134, 522066, 517192, 524092, 523132, 52206..."
2,CAM|0070|3900,517507,1010,"[523134, 522066, 517192, 524092, 523132, 52206..."
3,CAM|0070|49570,523133,1010,"[523134, 522066, 517192, 524092, 523132, 52206..."
4,CAM|0070|57010,523132,1010,"[523134, 522066, 517192, 524092, 523132, 52206..."


In [108]:
pan_rec = recomendaciones_validas[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [109]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0070|82880,522064
1,CAM|0070|58370,517507
2,CAM|0070|3900,517507
3,CAM|0070|49570,523133
4,CAM|0070|57010,523132


In [110]:
pan_rec.shape

(99484, 2)

In [111]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-8 Recomendar de acuerdo a Subida, Bajada, Mantener

In [112]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
#pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [113]:
pan_rec.shape

(99484, 2)

In [114]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [115]:
def desplazar_nulos_fila(fila):
    return pd.Series(sorted(fila, key=pd.isna), index=fila.index)

In [116]:
# Definir una función para asignar los valores subida, mantener y bajar
def clasificar_valor(x):
    if x > 0:
        return "S"
    elif x == 0:
        return "M"
    else:
        return "B"

In [117]:
# Definir la fecha de referencia (hoy)
fecha_actual = datetime.now(pytz.timezone("America/Lima"))#.strftime('%Y-%m-%d')
fecha_30dias_atras = (fecha_actual - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
fecha_60dias_atras = (fecha_actual - pd.Timedelta(days=60)).strftime('%Y-%m-%d')

print("fecha_actual",fecha_actual)
print("fecha_30dias_atras",fecha_30dias_atras)
print("fecha_60dias_atras",fecha_60dias_atras)

fecha_actual 2026-03-29 21:32:58.303942-05:00
fecha_30dias_atras 2026-02-27
fecha_60dias_atras 2026-01-28


In [118]:
# Filtrar los últimos 30 días
df_ultimos_30 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_30dias_atras) 
                          & (df_ventas['fecha_liquidacion'] <= fecha_actual.strftime('%Y-%m-%d'))]

# Filtrar los días 31 a 60
df_31_60 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_60dias_atras)
              & (df_ventas['fecha_liquidacion'] <= fecha_30dias_atras)]

# Agrupar por codigo_usuario, ruta, codigo_producto y sumar ventas
ventas_ultimos_30 = df_ultimos_30.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_ultimos_30["mes"] = "0_30"
ventas_31_60 = df_31_60.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_31_60["mes"] = "31_60"

df_grouped = pd.concat([ventas_ultimos_30,ventas_31_60],axis=0,ignore_index=True)
df_grouped = pd.pivot_table(
    df_grouped,
    values="imp_netovta",
    index=["cod_ruta", "cod_articulo_magic"],
    columns=["mes"],
    aggfunc="sum",
).reset_index()

# Rellenando con 0 si es nulo
df_grouped["0_30"] = df_grouped["0_30"].fillna(0)
df_grouped["31_60"] = df_grouped["31_60"].fillna(0)

# Asignando variacion de productos por mes
df_grouped["v1_2"] = (
    (df_grouped[df_grouped.columns[2]] - df_grouped[df_grouped.columns[3]])
    / df_grouped[df_grouped.columns[3]]
    * 100
)
df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)
df_grouped["v1_2"] = df_grouped["v1_2"].fillna(-1)
df_grouped["vp"] = (df_grouped["v1_2"])

/tmp/ipykernel_7255/261865676.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)


In [119]:
# Aplicar la función a la columna 'valor' y guardar el resultado en una nueva columna 'flag'
df_grouped["flag"] = df_grouped["vp"].apply(lambda x: clasificar_valor(x))
# Definir el mapeo de valores
mapeo_flag = {"S": 0, "M": 1, "B": 2}
# Aplicar el mapeo a la columna 'clasificacion' y guardar el resultado en una nueva columna 'flag_rank'
df_grouped["flag_rank"] = df_grouped["flag"].map(mapeo_flag)

In [120]:
df_grouped

mes,cod_ruta,cod_articulo_magic,0_30,31_60,v1_2,vp,flag,flag_rank
0,1001,500841,137168.1416,106194.6907,29.166666,29.166666,S,0
1,1001,500862,235619.4681,268805.3087,-12.345679,-12.345679,B,2
2,1001,501109,122787.6102,159292.0347,-22.916667,-22.916667,B,2
3,1001,501542,115044.2483,53097.3454,116.666667,116.666667,S,0
4,1001,502513,331858.4061,308628.3175,7.526882,7.526882,S,0
...,...,...,...,...,...,...,...,...
2353,29001,623645,3063.7167,5106.1946,-40.000001,-40.000001,B,2
2354,29001,623648,2042.4778,1021.2389,100.000000,100.000000,S,0
2355,29001,624499,5309.7346,5309.7346,0.000000,0.000000,M,1
2356,29001,624601,7128.7128,0.0000,-1.000000,-1.000000,B,2


In [121]:
pan_rec.shape

(99484, 3)

In [122]:
pan_rec = pd.merge(pan_rec,df_grouped[["cod_ruta", "cod_articulo_magic","flag_rank"]],on=["cod_ruta", "cod_articulo_magic"],how="left")

In [123]:
pan_rec["flag_rank"] = pan_rec["flag_rank"].fillna(3)

In [124]:
# Agregar una columna con el orden original
pan_rec["original_order"] = pan_rec.index

In [125]:
# Ordenar por cod_cliente, rank_producto (ascendente) y mantener el orden original en caso de empate
pan_rec = pan_rec.sort_values(by=["id_cliente", "flag_rank", "original_order"], ascending=[True, True, True])#.drop(columns=["original_order"])

In [126]:
pan_rec.head()

,id_cliente,cod_articulo_magic,cod_ruta,flag_rank,original_order
6271,CAM|0070|1022,523133,1008,0,6271
6334,CAM|0070|1022,523136,1008,0,6334
6397,CAM|0070|1022,523371,1008,0,6397
6460,CAM|0070|1022,599266,1008,0,6460
6647,CAM|0070|1022,517499,1008,0,6647


In [127]:
pan_rec.shape

(99484, 5)

In [128]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [129]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0070|1022,523133
1,CAM|0070|1022,523136
2,CAM|0070|1022,523371
3,CAM|0070|1022,599266
4,CAM|0070|1022,517499


In [130]:
pan_rec.shape

(99484, 2)

In [131]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-7 Usamos el archivo de validacion para no recomendar SKUs que no se deben a las rutas

In [132]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [133]:
pan_rec.shape

(99484, 2)

In [134]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [135]:
pan_rec.isnull().sum()

id_cliente            0
cod_articulo_magic    0
cod_ruta              0
dtype: int64

In [136]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

cod_ruta
1001    46
1002    50
1003    51
1004    41
1005    47
Name: cod_articulo_magic, dtype: int64

In [137]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

45.674418604651166

In [138]:
# Ruta del archivo en S3
s3_path = "s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/cam/maestro_productos_cam000"

# Leer el CSV directamente a un DataFrame
skus_val = pd.read_csv(s3_path,sep = ";")
# skus_val = pd.read_csv("maestra_productos_cam_20250217.csv",sep=";")
# skus_val.head()

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [139]:
skus_val = skus_val[skus_val.cod_compania==70]
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print("fecha_proceso: ", hoy)
# Filtrar solo las filas con la fecha de hoy
# skus_val = skus_val[skus_val['fecha_proceso'] == hoy]
skus_val["cod_compania"] = (skus_val["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
skus_val["id_cliente"] = (
    "CAM"
    + "|"
    + skus_val["cod_compania"].astype(str)
    + "|"
    + skus_val["cod_cliente"].astype(str)
)

fecha_proceso:  20260329


In [140]:
skus_val.head()

,cod_compania,desc_compania,cod_sucursal,desc_sucursal,cod_zona,desc_zona,cod_ruta,desc_ruta,cod_modulo,desc_modulo,cod_cliente,cod_articulo_magic,desc_articulo,id_cliente
0,0070,AJECEN DEL SUR S.A.,9,GUAPILES,6000,DETALLE,6002,DETALLE,6115,LU,65453,523137,BIG ROJA PET NO RETORNABLE 250 ML 12,CAM|0070|65453
1,0070,AJECEN DEL SUR S.A.,2,CEDIS CARTAGO,2000,DETALLE,1008,1008,1122,1122 L - J,82949,599266,VOLT GO RASPBERRY PET NO RETORNABLE 360 ML 12,CAM|0070|82949
2,0070,AJECEN DEL SUR S.A.,10,CORONADO,1250,CORONADO,1451,NaN,14507,NaN,81016,523134,BIG UVA PET NO RETORNABLE 250 ML 12,CAM|0070|81016
3,0070,AJECEN DEL SUR S.A.,9,GUAPILES,6000,DETALLE,6003,DETALLE,6116,JU,62983,523137,BIG ROJA PET NO RETORNABLE 250 ML 12,CAM|0070|62983
4,0070,AJECEN DEL SUR S.A.,10,CORONADO,1250,CORONADO,1450,NaN,14501,NaN,81521,523371,DILYTE FRESA KIWI PET NO RETORNABLE 650 ML 6,CAM|0070|81521


In [141]:
skus_val.shape

(250985, 14)

In [142]:
# skus_val = pd.read_csv("productos_pedido_sugerido_validacion_cam_090225.csv",sep=";")
# skus_val = skus_val[skus_val.cod_compania==70].reset_index(drop=True)

In [143]:
# Cada ruta pertenece a una compañia
skus_val.groupby("cod_ruta").cod_compania.nunique().mean()

1.0

In [144]:
pan_rec = pd.merge(pan_rec,skus_val[['cod_articulo_magic', 'id_cliente']].drop_duplicates(),left_on=["id_cliente","cod_articulo_magic"],right_on=["id_cliente","cod_articulo_magic"],how="inner")

In [145]:
pan_rec.shape

(42510, 3)

In [146]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

cod_ruta
1001    46
1002    50
1003    51
1004    41
1005    47
Name: cod_articulo_magic, dtype: int64

In [147]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

45.674418604651166

In [148]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

19.698795180722893

In [149]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]]

In [150]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0070|1022,523133
1,CAM|0070|1022,523136
2,CAM|0070|1022,523371
3,CAM|0070|1022,599266
4,CAM|0070|1022,517499


In [151]:
pan_rec.shape

(42510, 2)

In [152]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-6 Priorizar/mover SKUs para asegurar su recomendacion (Solo correr si se tienen SKUs a priorizar)

In [153]:
# pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [154]:
# skus_a_priorizar = [500580, 500713, 501022, 501505, 501603, 502501, 522398, 622061]

In [155]:
# pan_rec.shape

In [156]:
# # SKUs priorizados
# temp_1 = pan_rec[pan_rec.cod_articulo_magic.isin(skus_a_priorizar)]
# # SKUs NO priorizados
# temp_2 = pan_rec[~pan_rec.cod_articulo_magic.isin(skus_a_priorizar)]

In [157]:
# pan_rec = pd.concat([temp_1, temp_2], axis=0, ignore_index=True)

In [158]:
# pan_rec.shape

In [159]:
# pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-5 Filtrar SKUs por STOCK

In [160]:
stock = pd.read_csv("s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/cam/D_stock_cr.csv")

In [161]:
stock.drop(columns=["Fecha","Database"], inplace=True)
stock.columns=["cod_compania","cod_sucursal","cod_articulo_magic","stock_cf"]

In [162]:
stock["cod_compania"] = (stock["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
stock["cod_sucursal"] = (stock["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [163]:
fecha_hace_12_dias = (datetime.today() - timedelta(days=12)).strftime('%Y-%m-%d')
print(fecha_hace_12_dias)

2026-03-18


In [164]:
# Promedio diario de cajas fisicas por sucursal y compañia de los ultimos 12 dias
prom_diario_vta = df_ventas[df_ventas.cant_cajafisicavta > 0]
prom_diario_vta = df_ventas[df_ventas.fecha_liquidacion>=fecha_hace_12_dias].groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic","fecha_liquidacion"]).cant_cajafisicavta.sum().reset_index().groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic"]).cant_cajafisicavta.mean().reset_index()
prom_diario_vta["cod_compania"] = (prom_diario_vta["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
prom_diario_vta["cod_sucursal"] = (prom_diario_vta["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [165]:
df_stock = pd.merge(prom_diario_vta,stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"],how="left")
df_stock["dias_stock"] = df_stock["stock_cf"]/df_stock["cant_cajafisicavta"]
df_stock =  df_stock[(df_stock.dias_stock > 3) & (df_stock.cant_cajafisicavta > 0)].reset_index(drop=True)

In [166]:
pan_rec.nunique()

id_cliente            2158
cod_articulo_magic      74
dtype: int64

In [167]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

19.698795180722893

In [168]:
pan_rec.shape

(42510, 2)

In [169]:
# Cruzamos stock

In [170]:
test = pd.merge(pan_rec,df_ventas[["id_cliente","cod_compania","cod_sucursal"]].drop_duplicates(),on="id_cliente",how="left")

In [171]:
test["cod_compania"] = (test["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
test["cod_sucursal"] = (test["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [172]:
test_f = pd.merge(test, df_stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"], how="inner")

In [173]:
test_f.head()

,id_cliente,cod_articulo_magic,cod_compania,cod_sucursal,cant_cajafisicavta,stock_cf,dias_stock
0,CAM|0070|1022,523133,0070,02,115.666667,2871,24.821326
1,CAM|0070|1022,523136,0070,02,276.777778,4736,17.111200
2,CAM|0070|1022,523371,0070,02,9.888889,363,36.707865
3,CAM|0070|1022,599266,0070,02,96.555556,1902,19.698504
4,CAM|0070|1022,517499,0070,02,38.375000,1930,50.293160


In [174]:
test_f.shape

(39243, 7)

In [175]:
test_f.nunique()

id_cliente            2117
cod_articulo_magic      74
cod_compania             1
cod_sucursal             5
cant_cajafisicavta     178
stock_cf               194
dias_stock             261
dtype: int64

In [176]:
PS_VAL_promedio_recs_por_cliente_stock = (
    test_f.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_stock

18.537080774681154

In [177]:
pan_rec = test_f[["id_cliente","cod_articulo_magic"]].copy()

In [178]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-4 Mantener Solo SKUS en especifico (no correr sin CSV o XLSX de SKUs activos)

In [179]:
# pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# # pan_rec["id_cliente"]="CAM|"+pan_rec["id_cliente"]

In [180]:
# sku_con_precio = pd.read_excel(
#     "Input/CR - SKUS ACTIVOS_2024-10-31.xlsx", sheet_name="GTM CR - PRODUCTOS"
# )

In [181]:
# pan_rec = pan_rec[
#     pan_rec.cod_articulo_magic.isin(sku_con_precio["SKU"].unique())
# ].reset_index(drop=True)
# pan_rec

In [182]:
# # Size de recomendaciones luego de filtar solo a los del excel de SKUs activos
# pan_rec.shape

In [183]:
# PS_VAL_total_skus_recomendados_stock_y_activos = pan_rec["cod_articulo_magic"].nunique()
# PS_VAL_total_skus_recomendados_stock_y_activos

In [184]:
# pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-3 Quitar SKUS en especifico (no correr sin tener SKUs especificos)

In [185]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [186]:
# 622402 no deja vender por unidades
skus_sin_precio = [522065, 522064, 523138, 599328, 599329, 622108, 501638, 500713, 502501, 501505, 501603]

In [187]:
pan_rec = pan_rec[~(pan_rec["cod_articulo_magic"].isin(skus_sin_precio))].reset_index(
    drop=True
)

In [188]:
# PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda = (
#     pan_rec.cod_articulo_magic.nunique()
#  )
# PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda

In [189]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-2 Quitar Recomendaciones de los ultimos 2 semanas (no correr sin historial)

In [190]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"]="CAM|"+pan_rec["id_cliente"]

In [191]:
pan_rec

,id_cliente,cod_articulo_magic
0,CAM|0070|1022,523133
1,CAM|0070|1022,523136
2,CAM|0070|1022,523371
3,CAM|0070|1022,599266
4,CAM|0070|1022,517499
...,...,...
37356,CAM|0070|96890,523136
37357,CAM|0070|96890,523131
37358,CAM|0070|96890,523134
37359,CAM|0070|96890,523137


In [192]:
df_ventas=pd.read_parquet("Processed/cr_ventas_manana.parquet")

In [193]:
master_prod = pd.read_csv("Input/D_pan_masterProd.csv")

In [194]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-30'

In [195]:
ruta_archivo = f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv"
# Verificar si el archivo existe
if os.path.exists(ruta_archivo):
    # Eliminar el archivo si existe
    os.remove(ruta_archivo)
    print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
else:
    print(f"El archivo en la ruta {ruta_archivo} no existe.")

El archivo en la ruta Output/PS_piloto_v1/D_base_pedidos_2026-03-30.csv no existe.


In [196]:
# Directorio donde se encuentran los archivos
directorio = "Output/PS_piloto_v1/"

# Expresión regular para encontrar fechas en el nombre de los archivos
patron_fecha = r"\d{4}-\d{2}-\d{2}"

# Lista para almacenar las fechas extraídas
fechas = []

# Iterar sobre los archivos en el directorio
for archivo in os.listdir(directorio):
    # Verificar si el archivo es un archivo CSV y coincide con el patrón de nombres
    if archivo.endswith(".csv") and re.match(
        r"^D_base_pedidos_\d{4}-\d{2}-\d{2}\.csv$", archivo
    ):
        # Extraer la fecha del nombre del archivo
        fecha = re.search(patron_fecha, archivo).group()
        # Agregar la fecha a la lista
        fechas.append(fecha)

In [197]:
last_7_days_2 = sorted(fechas)[-14:]
last_7_days_2

['2026-03-13',
 '2026-03-14',
 '2026-03-16',
 '2026-03-17',
 '2026-03-18',
 '2026-03-19',
 '2026-03-20',
 '2026-03-21',
 '2026-03-23',
 '2026-03-24',
 '2026-03-25',
 '2026-03-26',
 '2026-03-27',
 '2026-03-28']

In [198]:
# Leer recomendaciones de los ultimos 7 dias y cruzar con clientes que tienen recomendacion para mañana
last_7_days_recs = pd.DataFrame()
for fecha_rec in last_7_days_2:
    df_temp = pd.read_csv(
        f"Output/PS_piloto_v1/D_base_pedidos_{fecha_rec}.csv",
        dtype={"Compania": "str", "Cliente": "str"},
    )
    df_temp["id_cliente"] = "CAM|" + df_temp["Compania"] + "|" + df_temp["Cliente"]
    df_temp = df_temp[df_temp["id_cliente"].isin(pan_rec["id_cliente"].unique())]
    last_7_days_recs = pd.concat([last_7_days_recs, df_temp], axis=0)
    print(f"{fecha_rec} done")

2026-03-13 done
2026-03-14 done
2026-03-16 done
2026-03-17 done
2026-03-18 done
2026-03-19 done
2026-03-20 done
2026-03-21 done
2026-03-23 done
2026-03-24 done
2026-03-25 done
2026-03-26 done
2026-03-27 done
2026-03-28 done


In [199]:
# Utilizamos merge para combinar los dataframes, utilizando la columna 'Cliente' como clave
df_combinado = pd.merge(
    pan_rec,
    last_7_days_recs,
    left_on=["id_cliente", "cod_articulo_magic"],
    right_on=["id_cliente", "Producto"],
    how="left",
    indicator=True,
)
# Filtramos los registros que solo están en el DataFrame 1
df_resultado = df_combinado[df_combinado["_merge"] == "left_only"][
    ["id_cliente", "cod_articulo_magic"]
].reset_index(drop=True)
df_resultado

,id_cliente,cod_articulo_magic
0,CAM|0070|1022,523133
1,CAM|0070|1022,523136
2,CAM|0070|1022,523371
3,CAM|0070|1022,599266
4,CAM|0070|1022,517499
...,...,...
28809,CAM|0070|96890,523136
28810,CAM|0070|96890,523131
28811,CAM|0070|96890,523134
28812,CAM|0070|96890,523137


In [200]:
df_resultado.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.0 Calcular Irregularidad de clientes

In [201]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [202]:
pan_rec["id_cliente"].nunique()

2078

In [203]:
now = pd.to_datetime(datetime.now(pytz.timezone("America/Lima")).strftime("%Y-%m-01"))
fecha_doce_meses_atras = now - pd.DateOffset(months=12)
lista_m12 = [fecha_doce_meses_atras + pd.DateOffset(months=i) for i in range(12)]
lista_m12

fecha_seis_meses_atras = now - pd.DateOffset(months=6)
lista_m6 = [fecha_seis_meses_atras + pd.DateOffset(months=i) for i in range(6)]
lista_m6

[Timestamp('2025-09-01 00:00:00'),
 Timestamp('2025-10-01 00:00:00'),
 Timestamp('2025-11-01 00:00:00'),
 Timestamp('2025-12-01 00:00:00'),
 Timestamp('2026-01-01 00:00:00'),
 Timestamp('2026-02-01 00:00:00')]

In [204]:
temp_df_ventas = df_ventas[df_ventas["id_cliente"].isin(pan_rec["id_cliente"].unique())]

In [205]:
qw = (
    temp_df_ventas[["id_cliente", "mes"]]
    .drop_duplicates()
    .sort_values(["id_cliente", "mes"])
    .groupby("id_cliente")
    .tail(12)
    .reset_index(drop=True)
)
qw["mes"] = pd.to_datetime(qw["mes"])
qw["m12"] = qw["mes"].isin(lista_m12)
qw["m6"] = qw["mes"].isin(lista_m6)

In [206]:
categoria_cliente = qw.groupby("id_cliente")[["m12", "m6"]].sum().reset_index()

In [207]:
# Definir condiciones
condicion_super = categoria_cliente["m12"] == 12
condicion_frecuente = (
    (categoria_cliente["m12"] < 12)
    & (categoria_cliente["m12"] >= 6)
    & (categoria_cliente["m6"] == 6)
)
condicion_regulares = (categoria_cliente["m6"] <= 6) & (categoria_cliente["m12"] <= 6)
condicion_riesgo = (categoria_cliente["m6"] < 6) & (categoria_cliente["m6"] >= 4)
condicion_irregulares = categoria_cliente["m6"] < 4

# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_super, "categoria_cliente"] = "Super"
categoria_cliente.loc[condicion_frecuente, "categoria_cliente"] = "Frecuente"
categoria_cliente.loc[condicion_regulares, "categoria_cliente"] = "Regular"
categoria_cliente.loc[condicion_riesgo, "categoria_cliente"] = "Riesgo"
categoria_cliente.loc[condicion_irregulares, "categoria_cliente"] = "Irregular"

**Recalcular Irregularidad**

In [208]:
# Definir condiciones
condicion_regular = (
    (categoria_cliente["categoria_cliente"] == "Super")
    | (categoria_cliente["categoria_cliente"] == "Frecuente")
    | (categoria_cliente["categoria_cliente"] == "Regular")
    | (categoria_cliente["categoria_cliente"] == "Riesgo")
)
condicion_irregular = categoria_cliente["categoria_cliente"] == "Irregular"
# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_regular, "categoria_cliente_2"] = "Regular"
categoria_cliente.loc[condicion_irregular, "categoria_cliente_2"] = "Irregular"

In [209]:
categoria_cliente

,id_cliente,m12,m6,categoria_cliente,categoria_cliente_2
0,CAM|0070|1022,11,6,Frecuente,Regular
1,CAM|0070|10925,11,6,Frecuente,Regular
2,CAM|0070|11615,11,6,Frecuente,Regular
3,CAM|0070|11619,11,6,Frecuente,Regular
4,CAM|0070|11652,11,6,Frecuente,Regular
...,...,...,...,...,...
2073,CAM|0070|96885,0,0,Irregular,Irregular
2074,CAM|0070|96886,0,0,Irregular,Irregular
2075,CAM|0070|96887,0,0,Irregular,Irregular
2076,CAM|0070|96888,0,0,Irregular,Irregular


In [210]:
categoria_cliente["categoria_cliente"].value_counts(dropna=False)

categoria_cliente
Frecuente    1462
Irregular     348
Riesgo        228
Regular        28
Super          12
Name: count, dtype: int64

In [211]:
categoria_cliente["categoria_cliente_2"].value_counts(dropna=False)

categoria_cliente_2
Regular      1730
Irregular     348
Name: count, dtype: int64

## 5.1. Obteniendo marcas de recomendaciones por cliente

In [212]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [213]:
pan_ventas = df_ventas[(df_ventas["cod_ruta"].isin(rutas))]
pan_ventas2 = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
pan_ventas = pd.concat([pan_ventas, pan_ventas2], axis=0)

In [214]:
marca_articulo = pan_ventas[["desc_categoria", "cod_articulo_magic"]].drop_duplicates()
print(marca_articulo.shape)
marca_articulo.head()

(84, 2)


,desc_categoria,cod_articulo_magic
0,GASEOSAS,522062
1,GASEOSAS,502513
2,GASEOSAS,522066
3,GASEOSAS,523135
4,ENERGIZANTE,599266


In [215]:
cliente_rec_marca = pd.merge(
    pan_rec, marca_articulo, on="cod_articulo_magic", how="left"
)
cliente_rec_marca["desc_categoria"] = cliente_rec_marca["desc_categoria"].str.strip()
print(cliente_rec_marca.shape)
cliente_rec_marca

(28814, 3)


,id_cliente,cod_articulo_magic,desc_categoria
0,CAM|0070|1022,523133,GASEOSAS
1,CAM|0070|1022,523136,JUGOS LIGEROS
2,CAM|0070|1022,523371,SUERO
3,CAM|0070|1022,599266,ENERGIZANTE
4,CAM|0070|1022,517499,ALOE VERA
...,...,...,...
28809,CAM|0070|96890,523136,JUGOS LIGEROS
28810,CAM|0070|96890,523131,JUGOS LIGEROS
28811,CAM|0070|96890,523134,GASEOSAS
28812,CAM|0070|96890,523137,GASEOSAS


## 5.2. Calcular marcas distintas recomendadas por cliente

In [216]:
# Conteo de categorias recomendadas por Cliente
cods2 = cliente_rec_marca.groupby("id_cliente")["desc_categoria"].nunique().reset_index()
print(cods2.shape)
cods2.head()

(2078, 2)


,id_cliente,desc_categoria
0,CAM|0070|1022,9
1,CAM|0070|10925,8
2,CAM|0070|11615,5
3,CAM|0070|11619,4
4,CAM|0070|11652,8


## 5.3 Quitando los SKUs de las ultimas 2 semanas (Evitar recompra)

In [217]:
last_2_weeks = pd.to_datetime(datetime.now(pytz.timezone("America/Lima"))) - pd.DateOffset(
    days=14
)
last_2_weeks

Timestamp('2026-03-15 21:33:19.158397-0500', tz='America/Lima')

In [218]:
# Asegurarse de que 'fecha_liquidacion' esté en formato datetime
pan_ventas["fecha_liquidacion"] = pd.to_datetime(pan_ventas["fecha_liquidacion"])
# Convertir 'fecha_liquidacion' a la misma zona horaria que 'last_year'
pan_ventas["fecha_liquidacion"] = pan_ventas["fecha_liquidacion"].dt.tz_localize(
    "America/Lima", nonexistent="NaT", ambiguous="NaT")

In [219]:
df_quitar_2_weeks = pan_ventas[pan_ventas["fecha_liquidacion"]>=last_2_weeks][["id_cliente","cod_articulo_magic"]].drop_duplicates().reset_index(drop=True)

In [220]:
# Hacemos un merge para identificar coincidencias
cliente_rec_sin4 = cliente_rec_marca.merge(df_quitar_2_weeks, on=['id_cliente', 'cod_articulo_magic'], how='left', indicator=True)
# Filtramos solo los registros que NO están en df_quitar_2_weeks
cliente_rec_sin4 = cliente_rec_sin4[cliente_rec_sin4['_merge'] == 'left_only'].drop(columns=['_merge'])

In [221]:
cliente_rec_marca.id_cliente.nunique()

2078

In [222]:
cliente_rec_sin4.id_cliente.nunique()

1985

In [223]:
cliente_rec_sin4

,id_cliente,cod_articulo_magic,desc_categoria
2,CAM|0070|1022,523371,SUERO
5,CAM|0070|1022,522066,GASEOSAS
7,CAM|0070|1022,500862,GASEOSAS
9,CAM|0070|1022,523137,GASEOSAS
10,CAM|0070|1022,522501,BEBIDAS DEPORTIVAS
...,...,...,...
28629,CAM|0070|96739,517192,ENERGIZANTE
28630,CAM|0070|96739,523135,GASEOSAS
28631,CAM|0070|96739,523134,GASEOSAS
28632,CAM|0070|96739,522867,JUGOS LIGEROS


## 5.4 Filtro para calcula antiguedad de clientes

In [224]:
pan_ventas = pan_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"})

In [225]:
pan_ventas["fecha_creacion"]=pan_ventas["fecha_creacion"].fillna(0).astype(int)

In [226]:
pan_ventas["fecha_creacion"] = pd.to_datetime(
    pan_ventas["fecha_creacion"], format="%Y%m%d", errors="coerce"
)

In [227]:
# Convertir la columna 'fecha_creacion' a tipo fecha
pan_ventas["fecha_creacion"] = pd.to_datetime(
    pan_ventas["fecha_creacion"], format="%Y%m%d"
)

In [228]:
# Definir una función para etiquetar los clientes según su fecha de creación
def etiquetar_cliente(fecha_creacion):
    if pd.isnull(fecha_creacion):
        return (
            "nf"  # Si la fecha de creación es nula, etiquetar como 'nf' (no encontrada)
        )
    else:
        hoy = datetime.now()  # Obtener la fecha actual
        hace_12_meses = hoy - timedelta(days=365)  # Calcular hace 12 meses
        if fecha_creacion >= hace_12_meses:
            return "new_client"  # Si la fecha de creación está dentro de los últimos 12 meses, etiquetar como 'new_client'
        else:
            return "old_client"  # Si la fecha de creación es anterior a los últimos 12 meses, etiquetar como 'old_client'

In [229]:
# Aplicar la función a la columna 'fecha_creacion' para crear la nueva columna 'tipo_cliente'
pan_ventas["antiguedad"] = pan_ventas["fecha_creacion"].apply(etiquetar_cliente)

In [230]:
pan_antiguedad_clientes = (
    pan_ventas[["id_cliente", "fecha_creacion", "antiguedad"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

## 5.4.2 Obteniendo datos necesarios para armar DF para subir a SalesForce

In [231]:
datos_para_salesforce = (
    pan_ventas.groupby(
        ["id_cliente", "cod_compania", "cod_sucursal", "cod_cliente", "cod_ruta"]
    )[
        [
            "id_cliente",
            "cod_compania",
            "cod_sucursal",
            "cod_cliente",
            "cod_modulo",
            "cod_ruta",
        ]
    ]
    .head(1)
    .reset_index(drop=True)
)

## 5.5 Juntando las Recomendaciones Finales
 - Ordenado de acuerdo al peso de la recomendación y el id de usuario.
 - Hay 5 skus de 5 marcas distintas que se recomiendan a cada usuario
 - Ningún SKU que haya comprado el usuario en sus ultimas X visitas será recomendado. La cantidad de visitas (X) a ignorar depende de sus visitas promedio mensuales.

In [232]:
# Obtener primer SKU de cada marca de la recomendacion
final_rec = cliente_rec_sin4.groupby(["id_cliente", "desc_categoria"]).first().reset_index()
# Agregar Marca de sus primeras compras
final_rec = pd.merge(final_rec, cods2, how="left", on="id_cliente")
# Agregar descripcion de SKU recomendado
pan_prod = pan_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
pan_prod = pan_prod.groupby(["cod_articulo_magic"]).first().reset_index()
final_rec = pd.merge(final_rec, pan_prod, how="left", on="cod_articulo_magic")
# Cambiar nombre de columnas
final_rec.columns = ["id_cliente", "marca_rec", "sku", "len_marca_rec", "desc_rec"]
# Agregar Giro del cliente
giros = df_ventas[df_ventas["id_cliente"].isin(final_rec["id_cliente"].unique())][
    ["id_cliente", "desc_giro", "desc_subgiro"]
].drop_duplicates()
final_rec = pd.merge(final_rec, giros, on="id_cliente", how="left")
# Agregar si el cliente es Regular o Irregular
final_rec = pd.merge(
    final_rec,
    categoria_cliente[["id_cliente", "categoria_cliente_2"]],
    on="id_cliente",
    how="left",
)
final_rec.loc[final_rec["categoria_cliente_2"] == "Irregular", "desc_giro"] = np.nan
# Agregar antiguedad de cliente
final_rec = pd.merge(final_rec, pan_antiguedad_clientes, how="left", on="id_cliente")
# Agregar segmento de cliente
info_segmentos = pan_ventas[
    [
        "id_cliente",
        "new_segment",
        "dias_de_visita__c",
        "periodo_de_visita__c",
        "ultima_visita",
    ]
].drop_duplicates()
info_segmentos = info_segmentos.groupby(["id_cliente"]).first().reset_index()
final_rec = pd.merge(final_rec, info_segmentos, how="left", on="id_cliente")
# Agregar datos necesarios para SALESFORCE
final_rec = pd.merge(final_rec, datos_para_salesforce, how="left", on="id_cliente")
# Agregar Peso de marcas dependiendo del giro del cliente
final_rec["peso"] = final_rec.apply(
    lambda row: mapeo_diccionario.get(row["desc_subgiro"], {}).get(row["marca_rec"], 5),
    axis=1)
# Ordenar la recomendacion de acuerdo al peso de la marca y nos quedamos con las 5 primeras marcas
final_rec = (
    final_rec.sort_values(["id_cliente", "peso"]).groupby(["id_cliente"]).head(5))
# Agregamos rank de la marca recomendada
final_rec["marca_rec_rank"] = final_rec.groupby("id_cliente").cumcount() + 1
final_rec = final_rec.reset_index(drop=True)

In [233]:
final_rec["sku"] = final_rec["sku"].astype(int)
final_rec["peso"] = final_rec["peso"].astype(int)
final_rec["marca_rec_rank"] = final_rec["marca_rec_rank"].astype(int)

In [234]:
final_rec

,id_cliente,marca_rec,sku,len_marca_rec,desc_rec,desc_giro,desc_subgiro,categoria_cliente_2,fecha_creacion,antiguedad,...,dias_de_visita__c,periodo_de_visita__c,ultima_visita,cod_compania,cod_sucursal,cod_cliente,cod_modulo,cod_ruta,peso,marca_rec_rank
0,CAM|0070|1022,GASEOSAS,522066,9,BIG COLA PET NO RETORNABLE 1000 ML 12,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,1;4,F1,2026-03-26,0070,2,1022,1122,1008,1,1
1,CAM|0070|1022,JUGOS LIGEROS,517507,9,BIO ALOE VERA NATURAL 500 ML 6 PACK,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,1;4,F1,2026-03-26,0070,2,1022,1122,1008,1,2
2,CAM|0070|1022,ENERGIZANTE,517192,9,VOLT FANTASY PET NO RETORNABLE 360 ml 12 pack,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,1;4,F1,2026-03-26,0070,2,1022,1122,1008,2,3
3,CAM|0070|1022,BEBIDAS DEPORTIVAS,522501,9,SPORADE BLUEBERRY PET NO RETORNABLE 625 ML 12,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,1;4,F1,2026-03-26,0070,2,1022,1122,1008,3,4
4,CAM|0070|1022,SUERO,523371,9,DILYTE FRESA KIWI PET NO RETORNABLE 650 ML 6,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,1;4,F1,2026-03-26,0070,2,1022,1122,1008,9,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7292,CAM|0070|96728,ALOE VERA,517499,7,BIO ALOE NATURAL PET NO RETORNABLE 360 ML 6,NaN,"BODEGAS, TIENDAS",Irregular,2026-03-12,new_client,...,1;4,F1,2026-03-26,0070,2,96728,1103,1003,3,4
7293,CAM|0070|96728,BEBIDAS DEPORTIVAS,522502,7,SPORADE FRUIT PUNCH PET NO RETORNABLE 625 ML 12,NaN,"BODEGAS, TIENDAS",Irregular,2026-03-12,new_client,...,1;4,F1,2026-03-26,0070,2,96728,1103,1003,3,5
7294,CAM|0070|96739,GASEOSAS,523135,3,BIG GINGER PET NO RETORNABLE 250 ML 12,NaN,PANADERIA/PASTELERIA,Irregular,2026-03-12,new_client,...,1;4,F1,2026-03-26,0070,2,96739,13521,1352,1,1
7295,CAM|0070|96739,JUGOS LIGEROS,522867,3,"CIFRUT NARANJA, MANDARINA Y LIMON PET NO RETOR...",NaN,PANADERIA/PASTELERIA,Irregular,2026-03-12,new_client,...,1;4,F1,2026-03-26,0070,2,96739,13521,1352,1,2


## 5.8. Guardando recomendacion para D&A y Comercial

In [235]:
final_rec.id_cliente.nunique()

1985

In [236]:
final_rec.dias_de_visita__c.value_counts(dropna=False)

dias_de_visita__c
1      4430
1;4    2867
Name: count, dtype: int64

In [237]:
final_rec.periodo_de_visita__c.value_counts(dropna=False)

periodo_de_visita__c
F1    7296
F2       1
Name: count, dtype: int64

### 5.8.1 Para D&A

In [238]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-30'

In [239]:
PS_VAL_total_clientes_con_visita_no_venta = final_rec.id_cliente.nunique()
PS_VAL_total_clientes_con_visita_no_venta

1985

In [240]:
final_rec["cod_cliente"] = final_rec["id_cliente"].str.split("|").str[-1]

In [241]:
PS_VAL_marcas_recomendadas = final_rec.marca_rec.unique()
PS_VAL_marcas_recomendadas

array(['GASEOSAS', 'JUGOS LIGEROS', 'ENERGIZANTE', 'BEBIDAS DEPORTIVAS',
       'SUERO', 'SNACKS', 'AGUA', 'NECTAR', 'SOPAS', 'ALOE VERA',
       'DETERGENTE', 'CEREALES'], dtype=object)

In [242]:
final_rec.to_csv(
    f"Output/PS_data_piloto_v1/D_pan_recs_data_{fecha_tomorrow}.csv", index=False
)

In [243]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_CostaRica/Output/PS_data_piloto_v1/D_pan_recs_data_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(final_rec, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_CostaRica/Output/PS_data_piloto_v1/D_pan_recs_data_2026-03-30.csv'],
 'partitions_values': {}}

### PRIORIZAR SKUs

In [244]:
# skus_a_priorizar = [523620, 523619, 517192]

In [245]:
# skus_a_priorizar

In [246]:
# skus_a_priorizar_p1 = pan_ventas[pan_ventas.cod_articulo_magic.isin(skus_a_priorizar)].groupby(
#     ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "cod_articulo_magic"]).cant_cajafisicavta.sum().reset_index().sort_values(
#     ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "cant_cajafisicavta"], ascending = [True,True,True,True,False]).reset_index(drop=True).groupby(
#     ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo"]).head(1).reset_index(drop=True)

In [247]:
# skus_a_priorizar_p1 = skus_a_priorizar_p1.drop('cant_cajafisicavta', axis=1)

In [248]:
# cli_faltantes = final_rec[~final_rec.cod_cliente.isin(skus_a_priorizar_p1.cod_cliente.astype(str).unique())].reset_index(drop=True)

In [249]:
# skus_a_priorizar_p2 = pan_ventas[pan_ventas.id_cliente.isin(cli_faltantes.id_cliente.unique())][["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo"]].drop_duplicates().reset_index(drop=True)

In [250]:
# # Para asignar un valor aleatorio a cada fila:
# skus_a_priorizar_p2['cod_articulo_magic'] = np.random.choice(skus_a_priorizar, size=len(skus_a_priorizar_p2))

In [251]:
# skus_a_priorizar_final = pd.concat([skus_a_priorizar_p1, skus_a_priorizar_p2], axis=0, ignore_index=True)

In [252]:
# skus_a_priorizar_final.rename(columns={'cod_articulo_magic': 'sku'}, inplace=True)

In [253]:
# skus_a_priorizar_p1.cod_cliente.nunique()

In [254]:
# skus_a_priorizar_p2.cod_cliente.nunique()

In [255]:
# skus_a_priorizar_final.cod_cliente.nunique()

In [256]:
# final_rec = pd.concat([skus_a_priorizar_final, final_rec], axis=0, ignore_index=True)

## 5.9 Filtro de Recomendaciones por Segmento de cliente

In [257]:
# Definir una función para filtrar las filas según el valor de 'new_segment'
def filtrar_segmento(group):
    segmento = group["new_segment"].iloc[
        0
    ]  # Obtener el valor del segmento para el grupo
    if segmento == "BLINDAR":
        return group.head(1)  # Si el segmento es 'Blindar', mantener la primera fila
    elif segmento == "MANTENER":
        return group.head(
            2
        )  # Si el segmento es 'Mantener', mantener las dos primeras filas
    elif segmento == "DESARROLLAR":
        return group.head(
            3
        )  # Si el segmento es 'Desarrollar', mantener las tres primeras filas
    elif segmento == "OPTIMIZAR":
        return group.head(
            4
        )  # Si el segmento es 'Optimizar', mantener las cuatro primeras filas
    else:
        return group  # Si el segmento no es ninguno de los especificados, mantener todas las filas

In [258]:
PS_VAL_promedio_recs_por_cliente_negocio = (
    final_rec.groupby("id_cliente")["sku"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_negocio

3.334508816120907

In [259]:
final_rec = (
    final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)
)

/tmp/ipykernel_7255/3325175710.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)


In [260]:
final_rec["cod_compania"] = final_rec["cod_compania"].astype(int)
final_rec["cod_sucursal"] = final_rec["cod_sucursal"].astype(int)
final_rec["cod_cliente"] = final_rec["cod_cliente"].astype(int)
final_rec["cod_modulo"] = final_rec["cod_modulo"].astype(int)
final_rec["sku"] = final_rec["sku"].astype(int)

## 5.10. Formateando para subir a SF

In [261]:
recomendaciones_para_salesforce = final_rec[
    ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "sku"]
]
recomendaciones_para_salesforce = recomendaciones_para_salesforce.drop_duplicates()
recomendaciones_para_salesforce["Pais"] = "CR"
recomendaciones_para_salesforce["Cajas"] = int(1)
recomendaciones_para_salesforce["Unidades"] = int(0)
recomendaciones_para_salesforce["Fecha"] = fecha_tomorrow
recomendaciones_para_salesforce = recomendaciones_para_salesforce[
    [
        "Pais",
        "cod_compania",
        "cod_sucursal",
        "cod_cliente",
        "cod_modulo",
        "sku",
        "Cajas",
        "Unidades",
        "Fecha",
    ]
]
recomendaciones_para_salesforce.columns = [
    "Pais",
    "Compania",
    "Sucursal",
    "Cliente",
    "Modulo",
    "Producto",
    "Cajas",
    "Unidades",
    "Fecha",
]

In [262]:
recomendaciones_para_salesforce["Compania"] = recomendaciones_para_salesforce[
    "Compania"
].apply(lambda x: str(int(x)).rjust(4, "0"))
recomendaciones_para_salesforce["Sucursal"] = recomendaciones_para_salesforce[
    "Sucursal"
].apply(lambda x: str(int(x)).rjust(2, "0"))

In [263]:
recomendaciones_para_salesforce.dtypes

Pais        object
Compania    object
Sucursal    object
Cliente      int64
Modulo       int64
Producto     int64
Cajas        int64
Unidades     int64
Fecha       object
dtype: object

In [264]:
rutas_cliente = df_ventas[["cod_cliente", "cod_ruta"]].drop_duplicates()

In [265]:
recomendaciones_para_salesforce.Sucursal.unique()

array(['02', '06', '07', '09', '10', '04'], dtype=object)

In [266]:
recomendaciones_para_salesforce.Compania.unique()

array(['0070'], dtype=object)

In [267]:
recomendaciones_para_salesforce.Cliente.nunique()

1985

In [268]:
recomendaciones_para_salesforce

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha
0,CR,0070,02,1022,1122,522066,1,0,2026-03-30
1,CR,0070,02,10925,1114,523132,1,0,2026-03-30
2,CR,0070,02,10925,1114,522867,1,0,2026-03-30
3,CR,0070,02,10925,1114,523620,1,0,2026-03-30
4,CR,0070,02,11615,1123,523134,1,0,2026-03-30
...,...,...,...,...,...,...,...,...,...
4156,CR,0070,02,96689,13511,523136,1,0,2026-03-30
4157,CR,0070,02,96728,1103,524463,1,0,2026-03-30
4158,CR,0070,02,96728,1103,517507,1,0,2026-03-30
4159,CR,0070,02,96739,13521,523135,1,0,2026-03-30


In [269]:
recomendaciones_para_salesforce.Cliente.nunique()

1985

In [270]:
PS_VAL_total_final_clientes_recomendados = (
    recomendaciones_para_salesforce.Cliente.nunique()
)
PS_VAL_total_final_clientes_recomendados

1985

In [271]:
PS_VAL_promedio_final_recomendaciones_por_cliente = (
    recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()
)
PS_VAL_promedio_final_recomendaciones_por_cliente

1.9506297229219143

In [272]:
recomendaciones_para_salesforce.groupby("Sucursal")["Producto"].nunique()

Sucursal
02    52
04    42
06    47
07    45
09    48
10    40
Name: Producto, dtype: int64

In [273]:
PS_VAL_total_final_skus_recomendados = (
    recomendaciones_para_salesforce.Producto.nunique()
)
PS_VAL_total_final_skus_recomendados

67

**Conteo de marcas recomendadas**

In [274]:
count = (
    final_rec.groupby("id_cliente")["marca_rec"]
    .apply(lambda x: "-".join((x)))
    .reset_index(name="marcas_ordenadas")
)
count = (
    count.groupby("marcas_ordenadas")["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
)

In [275]:
PS_VAL_total_combinacion_rec_por_marca = count.shape[0]
PS_VAL_total_combinacion_rec_por_marca

203

**Conteo de SKUs recomendadas**

In [276]:
count2 = (
    recomendaciones_para_salesforce.groupby("Cliente")["Producto"]
    .apply(lambda x: "-".join(x.astype(str)))
    .reset_index(name="marcas_ordenadas")
)
count2 = (
    count2.groupby("marcas_ordenadas")["Cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
)
count2["f"] = count2["Cliente"] / count2["Cliente"].sum()
count2["fa"] = count2["f"].cumsum()

In [277]:
PS_VAL_total_combinacion_rec_por_sku = count2.shape[0]
PS_VAL_total_combinacion_rec_por_sku

969

In [278]:
PS_VAL_total_combinacion_rec_por_sku_al_80 = count2[count2["fa"] <= 0.8].shape[0]
PS_VAL_total_combinacion_rec_por_sku_al_80

572

In [279]:
recomendaciones_para_salesforce["Cliente"] = recomendaciones_para_salesforce[
    "Cliente"
].astype(str)
rutas_cliente["cod_cliente"] = rutas_cliente["cod_cliente"].astype(str)

In [280]:
pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left",
)

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,cod_cliente,cod_ruta
0,CR,0070,02,1022,1122,522066,1,0,2026-03-30,1022,1008
1,CR,0070,02,10925,1114,523132,1,0,2026-03-30,10925,1012
2,CR,0070,02,10925,1114,522867,1,0,2026-03-30,10925,1012
3,CR,0070,02,10925,1114,523620,1,0,2026-03-30,10925,1012
4,CR,0070,02,11615,1123,523134,1,0,2026-03-30,11615,1009
...,...,...,...,...,...,...,...,...,...,...,...
4156,CR,0070,02,96689,13511,523136,1,0,2026-03-30,96689,1253
4157,CR,0070,02,96728,1103,524463,1,0,2026-03-30,96728,1003
4158,CR,0070,02,96728,1103,517507,1,0,2026-03-30,96728,1003
4159,CR,0070,02,96739,13521,523135,1,0,2026-03-30,96739,1352


In [281]:
PS_VAL_rutas_skus_clientes_por_sucursal = pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left",
)
PS_VAL_rutas_skus_clientes_por_sucursal = (
    PS_VAL_rutas_skus_clientes_por_sucursal.groupby("Sucursal")
    .agg({"cod_ruta": "nunique", "Cliente": "nunique", "Producto": "nunique"})
    .reset_index()
)
PS_VAL_rutas_skus_clientes_por_sucursal.columns = [
    "Sucursal",
    "cant_rutas",
    "cant_clientes",
    "cant_skus",
]
PS_VAL_rutas_skus_clientes_por_sucursal

,Sucursal,cant_rutas,cant_clientes,cant_skus
0,02,25,1132,52
1,04,4,145,42
2,06,5,180,47
3,07,4,172,45
4,09,5,247,48
5,10,9,310,40


In [282]:
print(f"RESUMEN PS para {fecha_tomorrow}")
print(f"*" * 20, "Metricas Iniciales", "*" * 20)
print(f"{'PS_VAL_total_clientes_venta:':<70}{PS_VAL_total_clientes_venta}")
print(f"{'PS_VAL_total_clientes_visita:':<70}{PS_VAL_total_clientes_visita}")
print(f"{'PS_VAL_total_clientes_visita_trad:':<70}{PS_VAL_total_clientes_visita_trad}")
print(f"{'PS_VAL_ultima_venta:':<70}{PS_VAL_ultima_venta}")
print(f"{'PS_VAL_ultima_visita:':<70}{PS_VAL_ultima_visita}")
print(f"{'PS_VAL_total_skus_venta:':<70}{PS_VAL_total_skus_venta}")
print(f"*" * 20, "Metricas del Proceso", "*" * 20)
print(f"{'PS_VAL_total_clientes_merged:':<70}{PS_VAL_total_clientes_merged}")
print(f"{'PS_VAL_total_fechas_unicas:':<70}{PS_VAL_total_fechas_unicas}")
print(f"{'PS_VAL_total_skus_merged:':<70}{PS_VAL_total_skus_merged}")
print(
    f"{'PS_VAL_total_clientes_imp_netovta_positivo:':<70}{PS_VAL_total_clientes_imp_netovta_positivo}"
)
print(
    f"{'PS_VAL_total_clientes_para_manhana:':<70}{PS_VAL_total_clientes_para_manhana}"
)
print(
    f"{'PS_VAL_total_skus_venta_para_manhana:':<70}{PS_VAL_total_skus_venta_para_manhana}"
)
# print(
#     f"{'PS_VAL_total_skus_recomendados_stock_y_activos:':<70}{PS_VAL_total_skus_recomendados_stock_y_activos}"
# )
# print(
#     f"{'PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda:':<70}{PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda}"
# )
print(f"*" * 20, "Metricas Finales", "*" * 20)
print(
    f"{'PS_VAL_total_final_skus_recomendados:':<70}{PS_VAL_total_final_skus_recomendados}"
)
print(
    f"{'PS_VAL_total_final_clientes_recomendados:':<70}{PS_VAL_total_final_clientes_recomendados}"
)
print(
    f"{'PS_VAL_promedio_final_recomendaciones_por_cliente:':<70}{PS_VAL_promedio_final_recomendaciones_por_cliente}"
)
print(
    f"{'PS_VAL_total_combinacion_rec_por_marca:':<70}{PS_VAL_total_combinacion_rec_por_marca}"
)
print(f"{'PS_VAL_marcas_recomendadas:':<70}{PS_VAL_marcas_recomendadas}")
print(
    f"{'PS_VAL_total_combinacion_rec_por_sku:':<70}{PS_VAL_total_combinacion_rec_por_sku}"
)
print(
    f"{'PS_VAL_total_combinacion_rec_por_sku_al_80:':<70}{PS_VAL_total_combinacion_rec_por_sku_al_80}"
)
print(
    "PS_VAL_rutas_skus_clientes_por_sucursal:\n",
    PS_VAL_rutas_skus_clientes_por_sucursal,
)

RESUMEN PS para 2026-03-30
******************** Metricas Iniciales ********************
PS_VAL_total_clientes_venta:                                          11226
PS_VAL_total_clientes_visita:                                         11129
PS_VAL_total_clientes_visita_trad:                                    11129
PS_VAL_ultima_venta:                                                  ['2026-03-27']
PS_VAL_ultima_visita:                                                 ['2026-03-28']
PS_VAL_total_skus_venta:                                              85
******************** Metricas del Proceso ********************
PS_VAL_total_clientes_merged:                                         10886
PS_VAL_total_fechas_unicas:                                           338
PS_VAL_total_skus_merged:                                             85
PS_VAL_total_clientes_imp_netovta_positivo:                           10886
PS_VAL_total_clientes_para_manhana:                                   2161
PS_V

In [283]:
recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()

1.9506297229219143

In [284]:
recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().value_counts()

Producto
1    801
2    645
3    375
4    164
Name: count, dtype: int64

In [285]:
asd = (
    recomendaciones_para_salesforce.groupby("Cliente")
    .agg({"Producto": ["count", "nunique"]})
    .reset_index()
)
asd.columns = ["Cliente", "count", "nunique"]
asd["dif"] = asd["count"] == asd["nunique"]
asd.sort_values(["dif", "count"])

,Cliente,count,nunique,dif
912,66949,2,1,False
918,66969,2,1,False
920,66982,2,1,False
922,67001,2,1,False
923,67011,2,1,False
...,...,...,...,...
1803,94863,4,4,True
1826,95097,4,4,True
1829,95147,4,4,True
1875,95671,4,4,True


In [286]:
asd.dif.value_counts()

dif
True     1784
False     201
Name: count, dtype: int64

In [287]:
asd.value_counts("count")

count
1    715
2    616
3    402
4    252
Name: count, dtype: int64

In [288]:
recomendaciones_para_salesforce.to_csv(
    f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv", index=False
)

In [289]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_CostaRica/Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(recomendaciones_para_salesforce, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_CostaRica/Output/PS_piloto_v1/D_base_pedidos_2026-03-30.csv'],
 'partitions_values': {}}

In [290]:
recomendaciones_para_salesforce.shape

(4161, 9)

In [291]:
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_ventas["cod_cliente"] = (
    pan_ventas["cod_cliente"].astype(str)
)

In [292]:
val = pd.merge(recomendaciones_para_salesforce,pan_prod,left_on="Producto",right_on="cod_articulo_magic",how="left")
val = pd.merge(val,pan_ventas[["cod_compania","cod_cliente","cod_ruta","desc_sucursal","fecha_proceso"]].drop_duplicates(),left_on=["Compania","Cliente"],right_on=["cod_compania","cod_cliente"],how="left")

In [293]:
val

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,cod_articulo_magic,desc_articulo,cod_compania,cod_cliente,cod_ruta,desc_sucursal,fecha_proceso
0,CR,0070,02,1022,1122,522066,1,0,2026-03-30,522066,BIG COLA PET NO RETORNABLE 1000 ML 12,0070,1022,1008,CEDIS CARTAGO,20260329
1,CR,0070,02,10925,1114,523132,1,0,2026-03-30,523132,BIG NARANJA PET NO RETORNABLE 250 ML 12,0070,10925,1012,CEDIS CARTAGO,20260329
2,CR,0070,02,10925,1114,522867,1,0,2026-03-30,522867,"CIFRUT NARANJA, MANDARINA Y LIMON PET NO RETOR...",0070,10925,1012,CEDIS CARTAGO,20260329
3,CR,0070,02,10925,1114,523620,1,0,2026-03-30,523620,VOLT FRUIT PUNCH PET NO RETORNABLE 360 ML 12,0070,10925,1012,CEDIS CARTAGO,20260329
4,CR,0070,02,11615,1123,523134,1,0,2026-03-30,523134,BIG UVA PET NO RETORNABLE 250 ML 12,0070,11615,1009,CEDIS CARTAGO,20260329
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4911,CR,0070,02,96689,13511,523136,1,0,2026-03-30,523136,"CIFRUT NARANJA, MANDARINA Y LIMON PET NO RETOR...",0070,96689,1253,CEDIS CARTAGO,20260329
4912,CR,0070,02,96728,1103,524463,1,0,2026-03-30,524463,KR COLA PET NO RETORNABLE 2600 ML 6,0070,96728,1003,CEDIS CARTAGO,20260329
4913,CR,0070,02,96728,1103,517507,1,0,2026-03-30,517507,BIO ALOE VERA NATURAL 500 ML 6 PACK,0070,96728,1003,CEDIS CARTAGO,20260329
4914,CR,0070,02,96739,13521,523135,1,0,2026-03-30,523135,BIG GINGER PET NO RETORNABLE 250 ML 12,0070,96739,1352,CEDIS CARTAGO,20260329


In [294]:
val.cod_cliente.nunique()

1985

In [295]:
fecha_hoy_val = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=0)
).strftime("%Y-%m-%d")
fecha_hoy_val

'2026-03-29'

In [296]:
# Se usa guarda este DF porque contiene info util para comercial
# Se activó cuando se requerian hacer validaciones
# val.to_csv(f"Val_comercial/D_CR_validacion_recomendaciones_{fecha_hoy_val}.csv",index=False)

### 5.8.3 Enviar Correo con Metricas de PS

In [297]:
# Define la zona horaria de Lima, Perú
lima_tz = pytz.timezone("America/Lima")

# Obtén la fecha y hora actual en la zona horaria de Lima
fecha_hoy_lima = datetime.now(lima_tz)
fecha_hoy = fecha_hoy_lima.strftime("%Y-%m-%d")
fecha_hoy

'2026-03-29'

In [298]:
# Resta un día para obtener la fecha de ayer
fecha_ayer_lima = fecha_hoy_lima - timedelta(days=1)

# Formatea la fecha en el formato año-mes-día
fecha_ayer = fecha_ayer_lima.strftime("%Y-%m-%d")
fecha_ayer

'2026-03-28'

In [299]:
# Obtener el día de la semana (1 es lunes, 7 es domingo)
dia_semana_hoy = (fecha_hoy_lima - timedelta(days=0)).weekday() + 1
dia_semana_hoy

7

In [300]:
# Si es lunes (1), restar 2 días para comparar
if dia_semana_hoy == 1:  # Lunes
    fecha_comparacion = fecha_ayer_lima - timedelta(days=2)
else:
    fecha_comparacion = fecha_ayer_lima

# Comprobar la fecha
if fecha_comparacion.strftime("%Y-%m-%d") == PS_VAL_ultima_venta[0]:
    mensaje_venta = f"Venta: PS listo. El archivo de ventas se encuentra actualizado hasta {fecha_comparacion.strftime('%Y-%m-%d')}."
else:
    mensaje_venta = f"Venta: PS FALTA. Actualizado hasta {PS_VAL_ultima_venta[0]}"

In [301]:
mensaje_venta

'Venta: PS FALTA. Actualizado hasta 2026-03-27'

In [302]:
# Si es lunes (1), restar 2 días para comparar
if dia_semana_hoy == 1:  # Lunes
    fecha_comparacion = fecha_ayer_lima - timedelta(days=2)
else:
    fecha_comparacion = fecha_ayer_lima

# Comprobar la fecha
if fecha_comparacion.strftime("%Y-%m-%d") == PS_VAL_ultima_visita[0]:
    mensaje_visita = f"Visita: PS listo. El archivo de visitas se encuentra actualizado hasta {fecha_comparacion.strftime('%Y-%m-%d')}."
else:
    mensaje_visita = f"Visita: PS FALTA. Actualizado hasta {PS_VAL_ultima_visita[0]}"

In [303]:
mensaje_visita

'Visita: PS listo. El archivo de visitas se encuentra actualizado hasta 2026-03-28.'

In [304]:
# Credenciales para correo
remitente = "david.porta@ajegroup.com"
password = "pxqn kmuw drlq fvmv"

# Destinatario
asunto = f"Estado y Métricas PS para {fecha_tomorrow}"
# Lista de destinatarios
destinatarios = [
    "david.porta@ajegroup.com",
    "wilmer.rodriguez@ajegroup.com",
    "masaru.gonzales@ajegroup.com",
    "gherald.barzola@ajegroup.com",
    "jorge.delgado.pe@csoluciones.pe",
    "melissa.cotrina@ajegroup.com"
]
#destinatarios = ["david.porta@ajegroup.com"]

# Cuerpo de Correo
msg = MIMEMultipart()
msg["From"] = remitente
msg["To"] = ", ".join(destinatarios)
msg["Subject"] = asunto

In [305]:
# Convertir el DataFrame a HTML si PS_VAL_rutas_skus_clientes_por_sucursal es un DataFrame
tabla_rutas_skus_clientes = PS_VAL_rutas_skus_clientes_por_sucursal.to_html(index=False)

# quitados
# <tr><td>SKUs con stock y activos:</td><td>{PS_VAL_total_skus_recomendados_stock_y_activos}</td></tr>
# <tr><td>SKUs con stock, activos y precio según el lambda:</td><td>{PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda}</td></tr>
# Agregar mensaje al cuerpo del correo en formato HTML
cuerpo_correo = f"""
<html>
<head>
    <style>
        table {{
            width: 50%;
            border-collapse: collapse;
        }}
        th, td {{
            padding: 10px;
            border: 1px solid black;
            text-align: left;
        }}
        th {{
            background-color: #f2f2f2;
        }}
    </style>
</head>
<body>
    <h2>COSTA RICA</h2>
    <h3>Status de subida de PS Costa Rica para {fecha_tomorrow}:</h3>
    <p>{mensaje_venta}</p>
    
    <h3>Pendientes:</h3>
    <p style="font-style: italic; color: #555;">Se pide apoyo a Masaru del Equipo Comercial para validar 
    que la subida de PS a Salesforce se realizó de manera satisfactoria.</p>
    
    <h3>Resumen PS Costa Rica para {fecha_tomorrow}:</h3>

    <h4>Métricas de Input</h4>
    <table>
        <tr><td>Clientes en Historico de Ventas:</td><td>{PS_VAL_total_clientes_venta}</td></tr>
        <tr><td>Clientes en archivo de Visitas:</td><td>{PS_VAL_total_clientes_visita}</td></tr>
        <tr><td>Clientes en archivo de Visitas y Canal Tradicional:</td><td>{PS_VAL_total_clientes_visita_trad}</td></tr>
        <tr><td>Fecha ultima venta liquidada:</td><td>{PS_VAL_ultima_venta}</td></tr>
        <tr><td>Fecha ultima visita:</td><td>{PS_VAL_ultima_visita}</td></tr>
        <tr><td>SKUs distintos en Historico de Ventas:</td><td>{PS_VAL_total_skus_venta}</td></tr>
    </table>

    <h4>Métricas del Proceso</h4>
    <table>
        <tr><td>Clientes combinados de ventas y visitas:</td><td>{PS_VAL_total_clientes_merged}</td></tr>
        <tr><td>Fechas de liquidacion unicas en ventas:</td><td>{PS_VAL_total_fechas_unicas}</td></tr>
        <tr><td>SKUs distintos en ventas y visitas:</td><td>{PS_VAL_total_skus_merged}</td></tr>
        <tr><td>Clientes con importe de venta positivo:</td><td>{PS_VAL_total_clientes_imp_netovta_positivo}</td></tr>
        <tr><td>Clientes para mañana:</td><td>{PS_VAL_total_clientes_para_manhana}</td></tr>
        <tr><td>SKUs para clientes de mañana:</td><td>{PS_VAL_total_skus_venta_para_manhana}</td></tr>
    </table>

    <h4>Métricas Finales</h4>
    <table>
        <tr><td>SKUs finales recomendados:</td><td>{PS_VAL_total_final_skus_recomendados}</td></tr>
        <tr><td>Clientes finales recomendados:</td><td>{PS_VAL_total_final_clientes_recomendados}</td></tr>
        <tr><td>Clientes finales recomendados + clientes con visita y sin venta:</td><td>{PS_VAL_total_clientes_con_visita_no_venta}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego del modelo:</td><td>{PS_VAL_promedio_recs_por_cliente_modelo}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de stock:</td><td>{PS_VAL_promedio_recs_por_cliente_stock}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de reglas de negocio:</td><td>{PS_VAL_promedio_recs_por_cliente_negocio}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de reducir skus por segmento:</td><td>{PS_VAL_promedio_final_recomendaciones_por_cliente}</td></tr>
        <tr><td>Combinaciones de marca en recomendaciones:</td><td>{PS_VAL_total_combinacion_rec_por_marca}</td></tr>
        <tr><td>Marcas recomendadas:</td><td>{PS_VAL_marcas_recomendadas}</td></tr>
        <tr><td>Combinaciones de SKUs en recomendaciones:</td><td>{PS_VAL_total_combinacion_rec_por_sku}</td></tr>
        <tr><td>Combinaciones de SKUs al 80% de clientes:</td><td>{PS_VAL_total_combinacion_rec_por_sku_al_80}</td></tr>
    </table>

    <h4>Cantidad de Rutas, SKUs y Clientes por Sucursal:</h4>
    {tabla_rutas_skus_clientes}

</body>
</html>
"""

msg.attach(MIMEText(cuerpo_correo, "html"))

In [306]:
# Login SMTP de Gmail
server = smtplib.SMTP("smtp.gmail.com", 587)
server.starttls()
server.login(remitente, password)

(235, b'2.7.0 Accepted')

In [307]:
# Enviar correo
try:
    server.sendmail(remitente, destinatarios, msg.as_string())
    server.quit()
    print("Correo enviado")
except smtplib.SMTPException as e:
    print(f"Error SMTP al enviar correo: {e}")
except Exception as e:
    print(f"Error general al enviar correo: {e}")
except:
    print("Error al enviar correo")

Correo enviado


In [308]:
# # Detener el monitoreo
# running = False
# monitor_thread.join()

# # Graficar consumo de RAM y CPU
# fig, ax1 = plt.subplots(figsize=(10, 5))

# ax1.set_xlabel("Tiempo (segundos)")
# ax1.set_ylabel("RAM usada (GB)", color="tab:blue")
# ax1.plot(timestamps, mem_usage_gb, marker="o", linestyle="-", color="tab:blue", label="RAM (GB)")
# ax1.tick_params(axis="y", labelcolor="tab:blue")

# ax2 = ax1.twinx()  # Crear segundo eje Y para CPU
# ax2.set_ylabel("CPU usada (%)", color="tab:red")
# ax2.plot(timestamps, cpu_usage_percent, marker="x", linestyle="--", color="tab:red", label="CPU (%)")
# ax2.tick_params(axis="y", labelcolor="tab:red")

# fig.suptitle("Consumo de CPU y RAM del Notebook")
# ax1.grid()
# fig.tight_layout()
# plt.show()

In [309]:
# Apaga el Kernel para luego poder usar los demás notebooks sin llenar la memoria
import IPython
app = IPython.get_ipython()
app.kernel.do_shutdown(False)  # True indica apagar completamente el kernel

{'status': 'ok', 'restart': False}